In [1]:
import os
import math
import random
import re
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import (
    EfficientNetV2S, EfficientNetB3,EfficientNetB2,
    ResNet50V2, DenseNet121,
    MobileNetV3Large, ConvNeXtSmall, Xception
)
from tensorflow.keras.applications import (
    efficientnet_v2 as efv2_lib,
    efficientnet   as ef_lib,
    resnet_v2      as rv2_lib,
    densenet       as dn_lib,
    mobilenet_v3   as mv3_lib,
    convnext       as cnx_lib,
Xception as xcp_lib
)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint, LambdaCallback, CSVLogger, ReduceLROnPlateau, TerminateOnNaN

)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.regularizers import l2
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import os, re
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

try:
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    print('Mixed precision enabled')
except Exception as exc:
    print('Mixed precision not enabled:', exc)

print('TensorFlow :', tf.__version__)
print('GPUs       :', tf.config.list_physical_devices('GPU'))

2026-05-17 11:27:23.515154: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779017243.542101     872 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779017243.551097     872 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779017243.573003     872 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779017243.573024     872 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779017243.573027     872 computation_placer.cc:177] computation placer alr

TensorFlow: 2.19.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
Mixed precision enabled
TensorFlow : 2.19.0
GPUs       : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


# Custom models

In [ ]:
import gc
import os
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

TRAIN_DIR = '/kaggle/input/datasets/ahmedgamall/emotion-detection/Training_data/Training_data'
TEST_DIR  = '/kaggle/input/datasets/ahmedgamall/emotion-detection/test/test'
SAVE_DIR  = '/kaggle/working/models'
os.makedirs(SAVE_DIR, exist_ok=True)

IMG_SIZE  = 96
BATCH     = 32
SEED      = 42
CLASSES   = sorted(os.listdir(TRAIN_DIR))
N_CLASSES = len(CLASSES)

print(f'Classes ({N_CLASSES}): {CLASSES}')
print(f'Image size : {IMG_SIZE}x{IMG_SIZE}x1  (grayscale)')
print(f'Batch size : {BATCH}')

In [ ]:
import gc
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

gc.collect()

# ── Step 1: Collect paths only ────────────────────────────────
all_paths, all_labels = [], []

for label, cls in enumerate(CLASSES):
    folder = os.path.join(TRAIN_DIR, cls)
    for fname in os.listdir(folder):
        path = os.path.join(folder, fname)
        if os.path.isfile(path):
            all_paths.append(path)
            all_labels.append(label)

all_paths  = np.array(all_paths)
all_labels = np.array(all_labels, dtype=np.int32)

print('Raw distribution:')
for i, cls in enumerate(CLASSES):
    print(f'  {cls:<12} {np.sum(all_labels == i)}')

# ── Step 2: Rebalance paths only ──────────────────────────────
TARGET           = 5500
balanced_indices = []

print('\nRebalancing:')
for cls_idx in range(N_CLASSES):
    cls_indices = np.where(all_labels == cls_idx)[0]
    current     = len(cls_indices)
    if current >= TARGET:
        chosen = np.random.choice(cls_indices, size=TARGET, replace=False)
        action = 'undersampled'
    else:
        extra  = TARGET - current
        chosen = np.concatenate([
            cls_indices,
            np.random.choice(cls_indices, size=extra, replace=True)
        ])
        action = 'oversampled'
    balanced_indices.append(chosen)
    print(f'  {CLASSES[cls_idx]:<12} {current:>5} → {TARGET} ({action})')

balanced_indices = np.concatenate(balanced_indices)
rng              = np.random.default_rng(SEED)
balanced_indices = rng.permutation(balanced_indices)

bal_paths  = all_paths[balanced_indices]
bal_labels = all_labels[balanced_indices]

del all_paths, all_labels, balanced_indices
gc.collect()

# ── Step 3: Train/val split on paths ──────────────────────────
paths_train, paths_val, labels_train, labels_val = train_test_split(
    bal_paths, bal_labels,
    test_size    = 0.20,
    random_state = SEED,
    stratify     = bal_labels
)

del bal_paths, bal_labels
gc.collect()

print(f'\nTrain paths : {len(paths_train)}')
print(f'Val paths   : {len(paths_val)}')

# ── Step 4: Class weights ─────────────────────────────────────
CLASS_WEIGHTS = dict(enumerate(
    compute_class_weight('balanced',
                         classes=np.unique(labels_train),
                         y=labels_train)
))
print('\nClass weights:')
for k, v in CLASS_WEIGHTS.items():
    print(f'  {CLASSES[k]:<12} {v:.4f}')

# ── Step 5: load_image — grayscale version ────────────────────
def load_image_gray(path, label):
    raw = tf.io.read_file(path)
    img = tf.image.decode_image(
        raw,
        channels          = 1,       # grayscale — key difference from pretrained
        expand_animations = False
    )
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE], method='area')
    img = tf.cast(img, tf.float32)
    lbl = tf.one_hot(label, N_CLASSES)
    return img, lbl

# ── Step 6: augment — grayscale version ───────────────────────
# Identical to pretrained augment() — only change: channels 3 → 1
def augment_gray(img, lbl):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, max_delta=0.25)
    img = tf.image.random_contrast(img, lower=0.8, upper=1.2)

    pad = int(IMG_SIZE * 0.15)
    img = tf.pad(img, [[pad, pad], [pad, pad], [0, 0]], mode='REFLECT')
    offset_h = tf.random.uniform([], 0, 2 * pad, dtype=tf.int32)
    offset_w = tf.random.uniform([], 0, 2 * pad, dtype=tf.int32)
    img = tf.image.crop_to_bounding_box(img, offset_h, offset_w, IMG_SIZE, IMG_SIZE)

    crop_size = tf.random.uniform(
        [], minval=int(IMG_SIZE * 0.80),
        maxval=IMG_SIZE, dtype=tf.int32
    )
    img = tf.image.random_crop(img, size=[crop_size, crop_size, 1])  # 1 not 3
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.clip_by_value(img, 0.0, 255.0)
    return img, lbl

# ── Step 7: tf.data pipeline ──────────────────────────────────
AUTOTUNE = tf.data.AUTOTUNE

train_ds = (
    tf.data.Dataset
    .from_tensor_slices((paths_train, labels_train))
    .shuffle(buffer_size=3000, seed=SEED, reshuffle_each_iteration=True)
    .map(load_image_gray, num_parallel_calls=AUTOTUNE)   # parallel load
    .map(augment_gray,    num_parallel_calls=AUTOTUNE)   # parallel augment
    .batch(BATCH)
    .prefetch(AUTOTUNE)                                   # prefetch next batch
)

val_ds = (
    tf.data.Dataset
    .from_tensor_slices((paths_val, labels_val))
    .map(load_image_gray, num_parallel_calls=AUTOTUNE)
    .batch(BATCH)
    .prefetch(AUTOTUNE)
)

# ── Step 8: Load val into RAM for callbacks ───────────────────
# 6600 × 96×96×1 ≈ 0.24 GB — very safe
print('\nLoading val set into RAM for callbacks...')
X_val_list, y_val_list = [], []

for path, label in zip(paths_val, labels_val):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        continue
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    X_val_list.append(img)
    y_val_list.append(label)

X_val = np.array(X_val_list, dtype=np.float32)[..., np.newaxis]  # (N, 96, 96, 1)
y_val = to_categorical(np.array(y_val_list), N_CLASSES)

del X_val_list, y_val_list
gc.collect()

STEPS_PER_EPOCH = len(paths_train) // BATCH

print(f'X_val shape  : {X_val.shape}')
print(f'X_val RAM    : ~{X_val.nbytes / 1e9:.2f} GB')
print(f'Steps/epoch  : {STEPS_PER_EPOCH}')
print('\nPipeline ready — 0 GB train images in RAM')

# ── Verification ──────────────────────────────────────────────
print('\nVerification:')
for batch_imgs, batch_lbls in train_ds.take(1):
    print(f'  Batch shape : {batch_imgs.shape}')
    print(f'  Dtype       : {batch_imgs.dtype}')
    print(f'  Pixel range : [{batch_imgs.numpy().min():.0f}, {batch_imgs.numpy().max():.0f}]')
    print(f'  Label shape : {batch_lbls.shape}')
    counts = batch_lbls.numpy().argmax(1)
    print(f'  Label dist  : {np.bincount(counts, minlength=N_CLASSES)}')

In [ ]:
import albumentations as A
from tensorflow.keras.optimizers import AdamW

CUSTOM_STEPS = len(paths_train) // BATCH

# ── Training Tracker ──────────────────────────────────────────
class TrainingTracker(tf.keras.callbacks.Callback):
    def __init__(self, phase_name=""):
        super().__init__()
        self.phase_name   = phase_name
        self.best_val_acc = 0.0
        self.best_epoch   = 0
        self._history     = []

    def on_epoch_end(self, epoch, logs=None):
        t_acc  = logs.get('accuracy', 0)
        v_acc  = logs.get('val_accuracy', 0)
        t_loss = logs.get('loss', 0)
        v_loss = logs.get('val_loss', 0)
        lr     = logs.get('learning_rate', 0)
        gap    = t_acc - v_acc

        if v_acc > self.best_val_acc:
            self.best_val_acc = v_acc
            self.best_epoch   = epoch + 1

        self._history.append({'t_acc': t_acc, 'v_acc': v_acc,
                               't_loss': t_loss, 'v_loss': v_loss})

        if gap > 0.08:
            status = "⚠  OVERFIT"
        elif t_acc < 0.50 and epoch >= 9:
            status = "⚠  UNDERFIT"
        elif (len(self._history) >= 2 and
              v_loss > self._history[-2]['v_loss'] * 1.20):
            status = "⚠  VAL SPIKE"
        elif (len(self._history) >= 5 and
              max(h['v_acc'] for h in self._history[-5:]) -
              min(h['v_acc'] for h in self._history[-5:]) < 0.002):
            status = "—  PLATEAU"
        else:
            status = "✓"

        print(f"  [{self.phase_name}] Ep {epoch+1:3d} | "
              f"train {t_acc:.4f}/{t_loss:.4f}  "
              f"val {v_acc:.4f}/{v_loss:.4f}  "
              f"gap {gap:+.3f}  lr {lr:.2e}  {status}")

        if v_acc >= self.best_val_acc:
            print(f"           ★ New best val_acc: {v_acc:.4f} (epoch {self.best_epoch})")

    def on_train_end(self, logs=None):
        print(f"\n  [{self.phase_name}] Best val_acc: "
              f"{self.best_val_acc:.4f} at epoch {self.best_epoch}\n")


# ── Grayscale RandAugment ─────────────────────────────────────
randaug_gray = A.Compose([
    A.OneOf([
        A.Equalize(p=1.0),
        A.Sharpen(alpha=(0.2, 0.5), p=1.0),
        A.GaussNoise(var_limit=(5, 20), p=1.0),
        A.Blur(blur_limit=3, p=1.0),
        A.RandomBrightnessContrast(brightness_limit=0.2,
                                   contrast_limit=0.2, p=1.0),
    ], p=0.60),
    A.HorizontalFlip(p=0.5),
])

def apply_randaug_batch_gray(X):
    """X: (N, H, W, 1) float32 0-255"""
    out = np.empty_like(X)
    for i in range(len(X)):
        img = X[i, :, :, 0].clip(0, 255).astype(np.uint8)
        img = randaug_gray(image=img)['image']
        out[i, :, :, 0] = img.astype(np.float32)
    return out


# ── Random Erasing ────────────────────────────────────────────
def random_erasing_batch(X, prob=0.3, sl=0.02, sh=0.25, r1=0.3):
    out  = X.copy()
    H, W = X.shape[1], X.shape[2]
    area = H * W
    for i in range(len(out)):
        if np.random.rand() > prob:
            continue
        for _ in range(10):
            erase_area = np.random.uniform(sl, sh) * area
            aspect     = np.random.uniform(r1, 1.0 / r1)
            eh = int(np.sqrt(erase_area * aspect))
            ew = int(np.sqrt(erase_area / aspect))
            if eh >= H or ew >= W:
                continue
            ey = np.random.randint(0, H - eh)
            ex = np.random.randint(0, W - ew)
            out[i, ey:ey+eh, ex:ex+ew, :] = np.mean(out[i])
            break
    return out


# ── MixUp + CutMix generator — same pattern as pretrained ─────
def mixed_generator_ds_gray(dataset, cw_vals, alpha=0.3, cutmix_prob=0.5):
    """
    Identical to pretrained mixed_generator_ds —
    only difference: apply_randaug_batch_gray instead of apply_randaug_batch
    tf.data already handled: flip, brightness, contrast, shift, zoom
    This adds:  RandAugment → MixUp/CutMix → RandomErasing
    """
    ds_iter = iter(dataset.repeat())
    while True:
        X1, y1 = next(ds_iter)
        X2, y2 = next(ds_iter)
        X1, y1 = X1.numpy(), y1.numpy()
        X2, y2 = X2.numpy(), y2.numpy()
        bs     = min(len(X1), len(X2))
        X1, y1 = X1[:bs], y1[:bs]
        X2, y2 = X2[:bs], y2[:bs]

        # RandAugment (grayscale)
        X1 = apply_randaug_batch_gray(X1)
        X2 = apply_randaug_batch_gray(X2)

        if np.random.rand() < cutmix_prob:
            lam   = np.random.beta(alpha, alpha)
            H, W  = X1.shape[1], X1.shape[2]
            cut_h = int(H * np.sqrt(1.0 - lam))
            cut_w = int(W * np.sqrt(1.0 - lam))
            cx    = np.random.randint(0, W)
            cy    = np.random.randint(0, H)
            x1_c  = np.clip(cx - cut_w // 2, 0, W)
            y1_c  = np.clip(cy - cut_h // 2, 0, H)
            x2_c  = np.clip(cx + cut_w // 2, 0, W)
            y2_c  = np.clip(cy + cut_h // 2, 0, H)
            X_mix = X1.copy()
            X_mix[:, y1_c:y2_c, x1_c:x2_c, :] = X2[:, y1_c:y2_c, x1_c:x2_c, :]
            lam_eff = 1.0 - (y2_c - y1_c) * (x2_c - x1_c) / (H * W)
            y_mix   = lam_eff * y1 + (1.0 - lam_eff) * y2
        else:
            lam     = np.random.beta(alpha, alpha)
            X_mix   = lam * X1 + (1 - lam) * X2
            y_mix   = lam * y1 + (1 - lam) * y2
            lam_eff = lam

        X_mix = random_erasing_batch(X_mix, prob=0.3)

        w1    = cw_vals[y1.argmax(axis=1)]
        w2    = cw_vals[y2.argmax(axis=1)]
        w_mix = (lam_eff * w1 + (1 - lam_eff) * w2).astype(np.float32)
        yield X_mix, y_mix, w_mix


# ── LR schedule ───────────────────────────────────────────────
def make_lr_schedule(lr_max, warmup_epochs, total_epochs, min_lr=1e-6):
    def schedule(epoch, lr):
        if epoch < warmup_epochs:
            return float(lr_max * (epoch + 1) / max(warmup_epochs, 1))
        progress = (epoch - warmup_epochs) / max(total_epochs - warmup_epochs, 1)
        cosine   = 0.5 * (1.0 + np.cos(np.pi * progress))
        return float(min_lr + (lr_max - min_lr) * cosine)
    return schedule


# ── Callbacks factory ─────────────────────────────────────────
def make_callbacks_custom(name, lr_max, warmup, total,
                           patience, phase_name="", min_lr=1e-7):
    ckpt_path = os.path.join(SAVE_DIR, f'{name}_best.keras')
    log_path  = os.path.join(SAVE_DIR, f'{name}_log.csv')
    return [
        TerminateOnNaN(),
        tf.keras.callbacks.LearningRateScheduler(
            make_lr_schedule(lr_max, warmup, total, min_lr), verbose=0),
        EarlyStopping(monitor='val_accuracy', mode='max', patience=patience,
                      restore_best_weights=True, verbose=0),
        ModelCheckpoint(ckpt_path, monitor='val_accuracy', mode='max',
                        save_best_only=True, verbose=1),
        CSVLogger(log_path),
        TrainingTracker(phase_name=phase_name),
    ]


# ── TTA (grayscale) ───────────────────────────────────────────
def predict_with_tta_custom(model, X, n_aug=6):
    """X: (N, H, W, 1) float32 0-255"""
    tta_aug = ImageDataGenerator(
        horizontal_flip  = True,
        zoom_range       = 0.08,
        brightness_range = [0.88, 1.12],
        fill_mode        = 'reflect'
    )
    preds = model.predict(X, verbose=0)
    for _ in range(n_aug - 1):
        aug_gen = tta_aug.flow(X, batch_size=len(X), shuffle=False)
        X_aug   = next(aug_gen).clip(0, 255)
        preds  += model.predict(X_aug, verbose=0)
    return preds / n_aug


print('Custom utilities ready.')
print(f'Steps per epoch : {CUSTOM_STEPS}')
print('Augmentation flow:')
print('  tf.data  → flip / brightness / contrast / shift / zoom  (parallel)')
print('  generator → RandAugment → MixUp/CutMix → RandomErasing  (per batch)')

# Custom models

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers


def se_block(x, filters, ratio=16):
    se = layers.GlobalAveragePooling2D()(x)
    se = layers.Dense(max(filters // ratio, 1), activation='relu')(se)
    se = layers.Dense(filters, activation='sigmoid')(se)
    se = layers.Reshape((1, 1, filters))(se)
    return layers.Multiply()([x, se])


def residual_conv_block(x, filters, num_convs=3, dropout_rate=0.15):  # was 0.30
    shortcut = layers.Conv2D(filters, 1, padding='same', use_bias=False)(x)
    shortcut = layers.BatchNormalization()(shortcut)
    for _ in range(num_convs):
        x = layers.Conv2D(filters, 3, padding='same', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
    x = layers.Add()([x, shortcut])
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(dropout_rate)(x)
    return x


def build_deep_cnn(input_shape=(96, 96, 1), num_classes=6):
    inp = layers.Input(shape=input_shape)
    x   = layers.Rescaling(1./255)(inp)

    # Block 1
    x = layers.Conv2D(32, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(32, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.10)(x)  # was 0.25

    # Block 2
    x = layers.Conv2D(64, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(64, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.10)(x)  # was 0.25

    # Block 3
    x = layers.Conv2D(128, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(128, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(128, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.15)(x)  # was 0.30

    # Block 4 — residual
    x = residual_conv_block(x, filters=256, num_convs=3, dropout_rate=0.15)  # was 0.30

    # Block 5 — residual + SE
    shortcut5 = layers.Conv2D(512, 1, padding='same', use_bias=False)(x)
    shortcut5 = layers.BatchNormalization()(shortcut5)
    x = layers.Conv2D(512, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(512, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Add()([x, shortcut5])
    x = se_block(x, filters=512, ratio=16)

    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.Dense(512, activation='relu',
                       kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x   = layers.Dropout(0.30)(x)  # was 0.50
    x   = layers.Dense(256, activation='relu')(x)
    x   = layers.Dropout(0.15)(x)  # was 0.30
    out = layers.Dense(num_classes, activation='softmax')(x)

    return tf.keras.Model(inp, out, name='CustomDeepCNN')


model1 = build_deep_cnn(input_shape=(IMG_SIZE, IMG_SIZE, 1), num_classes=len(CLASSES))
model1.summary()


def train_deep_cnn(model):
    NAME    = 'CustomDeepCNN'
    smooth  = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05)  # was 0.10
    hard    = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.00)
    cw_vals = np.array([CLASS_WEIGHTS[i] for i in range(len(CLASSES))], dtype=np.float32)

    print('\n==== CustomDeepCNN | Phase 1 — AdamW peak LR=4e-4  70 ep ====')
    model.compile(
        optimizer = AdamW(learning_rate=4e-4, weight_decay=8e-5, clipnorm=1.0),
        loss      = smooth,
        metrics   = ['accuracy']
    )
    h1 = model.fit(
        mixed_generator_ds_gray(train_ds, cw_vals, alpha=0.2, cutmix_prob=0.5),
        steps_per_epoch = STEPS_PER_EPOCH,
        validation_data = (X_val, y_val),
        epochs          = 70,
        callbacks       = make_callbacks_custom(
                              NAME + '_p1', 4e-4, 5, 70, 10,
                              phase_name='P1', min_lr=1e-6),
        verbose=1
    )

    print('\n==== CustomDeepCNN | Phase 2 — AdamW peak LR=5e-5  35 ep ====')
    model.compile(
        optimizer = AdamW(learning_rate=5e-5, weight_decay=5e-5, clipnorm=1.0),
        loss      = hard,
        metrics   = ['accuracy']
    )
    h2 = model.fit(
        mixed_generator_ds_gray(train_ds, cw_vals, alpha=0.2, cutmix_prob=0.5),
        steps_per_epoch = STEPS_PER_EPOCH,
        validation_data = (X_val, y_val),
        epochs          = 35,
        callbacks       = make_callbacks_custom(
                              NAME + '_p2', 5e-5, 3, 35, 10,
                              phase_name='P2', min_lr=1e-7),
        verbose=1
    )

    best_p1 = max(h1.history['val_accuracy'])
    best_p2 = max(h2.history['val_accuracy'])
    best    = max(best_p1, best_p2)
    print(f'\n  Phase 1 best: {best_p1*100:.2f}%  |  Phase 2 best: {best_p2*100:.2f}%')
    print(f'  Overall best: {best*100:.2f}%')

    print('\n  Running TTA (6 passes)...')
    tta_preds = predict_with_tta_custom(model, X_val, n_aug=6)
    tta_acc   = np.mean(np.argmax(tta_preds, axis=1) == np.argmax(y_val, axis=1))
    print(f'  CustomDeepCNN TTA val accuracy: {tta_acc*100:.2f}%')

    model.save(os.path.join(SAVE_DIR, f'{NAME}_final.keras'))
    return best


acc1 = train_deep_cnn(model1)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers


def se_block_resnet(x, ratio=16):
    filters = x.shape[-1]
    se = layers.GlobalAveragePooling2D()(x)
    se = layers.Dense(max(filters // ratio, 8), activation='relu')(se)
    se = layers.Dense(filters, activation='sigmoid')(se)
    se = layers.Reshape((1, 1, filters))(se)
    return layers.Multiply()([x, se])


def residual_block(x, filters, stride=1, use_se=True, dropout_rate=0.0):
    shortcut = x
    out = layers.BatchNormalization()(x)
    out = layers.Activation('relu')(out)
    out = layers.Conv2D(filters, 3, strides=stride, padding='same', use_bias=False)(out)
    out = layers.BatchNormalization()(out)
    out = layers.Activation('relu')(out)
    out = layers.Conv2D(filters, 3, padding='same', use_bias=False)(out)
    if dropout_rate > 0.0:
        out = layers.SpatialDropout2D(dropout_rate)(out)
    if use_se:
        out = se_block_resnet(out)
    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides=stride,
                                 padding='same', use_bias=False)(shortcut)
    return layers.Add()([out, shortcut])


def build_custom_resnet(input_shape=(96, 96, 1), num_classes=6):
    inp = layers.Input(shape=input_shape)

    x = layers.Conv2D(32, 3, strides=1, padding='same', use_bias=False)(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(64, 3, strides=2, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    x = residual_block(x, 64,  use_se=True, dropout_rate=0.01)   # was 0.02
    x = residual_block(x, 64,  use_se=True, dropout_rate=0.01)   # was 0.02

    x = residual_block(x, 128, stride=2, use_se=True, dropout_rate=0.02)  # was 0.04
    x = residual_block(x, 128, use_se=True, dropout_rate=0.02)            # was 0.04
    x = residual_block(x, 128, use_se=True, dropout_rate=0.02)            # was 0.04

    x = residual_block(x, 256, stride=2, use_se=True, dropout_rate=0.03)  # was 0.06
    x = residual_block(x, 256, use_se=True, dropout_rate=0.03)            # was 0.06
    x = residual_block(x, 256, use_se=True, dropout_rate=0.03)            # was 0.06
    x = residual_block(x, 256, use_se=True, dropout_rate=0.03)            # was 0.06

    x = residual_block(x, 512, stride=2, use_se=True, dropout_rate=0.04)  # was 0.08
    x = residual_block(x, 512, use_se=True, dropout_rate=0.04)            # was 0.08

    x   = layers.BatchNormalization()(x)
    x   = layers.Activation('relu')(x)
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.Dense(256, activation='relu',
                       kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x   = layers.Dropout(0.25)(x)   # was 0.50
    x   = layers.Dense(128, activation='relu',
                       kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x   = layers.Dropout(0.15)(x)   # was 0.30
    out = layers.Dense(num_classes, activation='softmax')(x)

    return tf.keras.Model(inp, out, name='CustomResNet_v3')


model2 = build_custom_resnet(input_shape=(IMG_SIZE, IMG_SIZE, 1),
                              num_classes=len(CLASSES))
model2.summary()


def train_custom_resnet(model):
    NAME    = 'CustomResNet'
    smooth  = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.06)   # was 0.12
    refine  = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.03)   # was 0.06
    cw_vals = np.array([CLASS_WEIGHTS[i] for i in range(len(CLASSES))], dtype=np.float32)

    print('\n' + '='*70)
    print('  CustomResNet | Phase 1 — AdamW  peak LR=3e-4  50 ep  patience=12')
    print('='*70)
    model.compile(
        optimizer = AdamW(learning_rate=3e-4, weight_decay=8e-5, clipnorm=1.0),
        loss      = smooth,
        metrics   = ['accuracy']
    )
    h1 = model.fit(
        mixed_generator_ds_gray(train_ds, cw_vals, alpha=0.2, cutmix_prob=0.5),
        steps_per_epoch = STEPS_PER_EPOCH,
        validation_data = (X_val, y_val),
        epochs          = 50,
        callbacks       = make_callbacks_custom(
                              NAME + '_p1', 3e-4, 6, 50, 12,
                              phase_name='P1', min_lr=1e-6),
        verbose=1
    )

    print('\n' + '='*70)
    print('  CustomResNet | Phase 2 — AdamW  peak LR=4e-5  30 ep  patience=10')
    print('='*70)
    model.compile(
        optimizer = AdamW(learning_rate=4e-5, weight_decay=5e-5, clipnorm=1.0),
        loss      = refine,
        metrics   = ['accuracy']
    )
    h2 = model.fit(
        mixed_generator_ds_gray(train_ds, cw_vals, alpha=0.2, cutmix_prob=0.5),
        steps_per_epoch = STEPS_PER_EPOCH,
        validation_data = (X_val, y_val),
        epochs          = 30,
        callbacks       = make_callbacks_custom(
                              NAME + '_p2', 4e-5, 2, 30, 10,
                              phase_name='P2', min_lr=1e-7),
        verbose=1
    )

    best_p1 = max(h1.history['val_accuracy'])
    best_p2 = max(h2.history['val_accuracy'])
    best    = max(best_p1, best_p2)
    print('\n' + '='*70)
    print(f'  Phase 1 best: {best_p1*100:.2f}%  |  Phase 2 best: {best_p2*100:.2f}%')
    print(f'  Overall best: {best*100:.2f}%')
    print('='*70)

    print('\n  Running TTA (6 passes)...')
    tta_preds = predict_with_tta_custom(model, X_val, n_aug=6)
    tta_acc   = np.mean(np.argmax(tta_preds, axis=1) == np.argmax(y_val, axis=1))
    print(f'  CustomResNet TTA val accuracy: {tta_acc*100:.2f}%')

    model.save(os.path.join(SAVE_DIR, f'{NAME}_final.keras'))
    return best


acc2 = train_custom_resnet(model2)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers


def se_block_seresnet(x, reduction=16):
    filters = x.shape[-1]
    se = layers.GlobalAveragePooling2D()(x)
    se = layers.Reshape((1, 1, filters))(se)
    se = layers.Conv2D(max(filters // reduction, 8), 1, activation='relu')(se)
    se = layers.Conv2D(filters, 1, activation='sigmoid')(se)
    return layers.Multiply()([x, se])


def preact_conv_bn_relu(x, filters, kernel=3, stride=1):
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(filters, kernel, strides=stride,
                      padding='same', use_bias=False)(x)
    return x


def preact_se_residual_block(x, filters, stride=1, spatial_dropout=0.0):
    shortcut = x
    out = preact_conv_bn_relu(x, filters, stride=stride)
    out = preact_conv_bn_relu(out, filters)
    out = se_block_seresnet(out)
    if spatial_dropout > 0.0:
        out = layers.SpatialDropout2D(spatial_dropout)(out)
    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides=stride,
                                 padding='same', use_bias=False)(shortcut)
    return layers.Add()([out, shortcut])


def build_se_resnet(input_shape=(96, 96, 1), num_classes=6):
    inp = layers.Input(shape=input_shape)

    x = layers.Conv2D(32, 3, strides=1, padding='same', use_bias=False)(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(64, 3, strides=2, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    x = preact_se_residual_block(x, 64,  spatial_dropout=0.01)          # was 0.02
    x = preact_se_residual_block(x, 64,  spatial_dropout=0.01)          # was 0.02

    x = preact_se_residual_block(x, 128, stride=2, spatial_dropout=0.0)
    x = preact_se_residual_block(x, 128, spatial_dropout=0.02)          # was 0.04
    x = preact_se_residual_block(x, 128, spatial_dropout=0.02)          # was 0.04

    x = preact_se_residual_block(x, 256, stride=2, spatial_dropout=0.0)
    x = preact_se_residual_block(x, 256, spatial_dropout=0.03)          # was 0.06
    x = preact_se_residual_block(x, 256, spatial_dropout=0.03)          # was 0.06

    x = preact_se_residual_block(x, 512, stride=2, spatial_dropout=0.0)
    x = preact_se_residual_block(x, 512, spatial_dropout=0.04)          # was 0.08

    x   = layers.BatchNormalization()(x)
    x   = layers.Activation('relu')(x)
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.Dense(256, activation='relu',
                       kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x   = layers.Dropout(0.25)(x)   # was 0.50
    x   = layers.Dense(128, activation='relu',
                       kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x   = layers.Dropout(0.15)(x)   # was 0.30
    out = layers.Dense(num_classes, activation='softmax')(x)

    return tf.keras.Model(inp, out, name='CustomSEResNet_v2')


model4 = build_se_resnet(input_shape=(IMG_SIZE, IMG_SIZE, 1),
                         num_classes=len(CLASSES))
model4.summary()


def train_se_resnet(model):
    NAME    = 'CustomSEResNet'
    smooth  = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.06)   # was 0.12
    refine  = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.03)   # was 0.08
    cw_vals = np.array([CLASS_WEIGHTS[i] for i in range(len(CLASSES))], dtype=np.float32)

    print('\n' + '='*70)
    print('  CustomSEResNet | Phase 1 — AdamW  peak LR=2e-4  50 ep  patience=12')
    print('='*70)
    model.compile(
        optimizer = AdamW(learning_rate=2e-4, weight_decay=8e-5, clipnorm=1.0),
        loss      = smooth,
        metrics   = ['accuracy']
    )
    h1 = model.fit(
        mixed_generator_ds_gray(train_ds, cw_vals,
                              alpha=0.2, cutmix_prob=0.5),
        steps_per_epoch = STEPS_PER_EPOCH,
        validation_data = (X_val, y_val),
        epochs          = 50,
        callbacks       = make_callbacks_custom(
                              NAME + '_p1', 2e-4, 8, 50, 12,
                              phase_name='P1', min_lr=1e-6),
        verbose=1
    )

    print('\n' + '='*70)
    print('  CustomSEResNet | Phase 2 — AdamW  peak LR=4e-5  30 ep  patience=10')
    print('='*70)
    model.compile(
        optimizer = AdamW(learning_rate=4e-5, weight_decay=5e-5, clipnorm=1.0),
        loss      = refine,
        metrics   = ['accuracy']
    )
    h2 = model.fit(
        mixed_generator_ds_gray(train_ds, cw_vals,
                              alpha=0.2, cutmix_prob=0.5),
        steps_per_epoch = STEPS_PER_EPOCH,
        validation_data = (X_val, y_val),
        epochs          = 30,
        callbacks       = make_callbacks_custom(
                              NAME + '_p2', 4e-5, 2, 30, 10,
                              phase_name='P2', min_lr=1e-7),
        verbose=1
    )

    best_p1 = max(h1.history['val_accuracy'])
    best_p2 = max(h2.history['val_accuracy'])
    best    = max(best_p1, best_p2)
    print('\n' + '='*70)
    print(f'  Phase 1 best: {best_p1*100:.2f}%  |  Phase 2 best: {best_p2*100:.2f}%')
    print(f'  Overall best: {best*100:.2f}%')
    print('='*70)

    print('\n  Running TTA (6 passes)...')
    tta_preds = predict_with_tta_custom(model, X_val, n_aug=6)
    tta_acc   = np.mean(np.argmax(tta_preds, axis=1) == np.argmax(y_val, axis=1))
    print(f'  CustomSEResNet TTA val accuracy: {tta_acc*100:.2f}%')

    model.save(os.path.join(SAVE_DIR, f'{NAME}_final.keras'))
    return best


acc4 = train_se_resnet(model4)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CustomDWNet v3 — adapted for mixed_generator_ds_gray
# ══════════════════════════════════════════════════════════════════════════════

def se_block_dw(x, ratio=8):
    filters = x.shape[-1]
    se = layers.GlobalAveragePooling2D()(x)
    se = layers.Dense(max(filters // ratio, 16), activation='relu')(se)
    se = layers.Dense(filters, activation='sigmoid')(se)
    se = layers.Reshape((1, 1, filters))(se)
    return layers.Multiply()([x, se])


def dw_sep_block(x, filters, stride=1, use_se=True, use_residual=True,
                 spatial_dropout=0.0):
    residual = x

    out = layers.DepthwiseConv2D(3, strides=stride, padding='same',
                                 use_bias=False)(x)
    out = layers.BatchNormalization()(out)
    out = layers.Activation('relu')(out)

    out = layers.Conv2D(filters, 1, use_bias=False)(out)
    out = layers.BatchNormalization()(out)

    if use_se:
        out = se_block_dw(out)

    if spatial_dropout > 0.0:
        out = layers.SpatialDropout2D(spatial_dropout)(out)

    if use_residual and stride == 1 and residual.shape[-1] == filters:
        out = layers.Add()([out, residual])

    out = layers.Activation('relu')(out)
    return out


def build_custom_dw_cnn(input_shape=(96, 96, 1), num_classes=6):
    inp = layers.Input(shape=input_shape)

    # Stem: 96×96 → 48×48
    x = layers.Conv2D(32, 3, strides=2, padding='same', use_bias=False)(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(32, 3, strides=1, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # Stage 1: 48×48, 64 filters
    x = dw_sep_block(x, 64,  use_se=True, spatial_dropout=0.01)          # was 0.02

    # Stage 2: 24×24, 128 filters
    x = dw_sep_block(x, 128, stride=2, use_se=True, use_residual=False,
                     spatial_dropout=0.0)
    x = dw_sep_block(x, 128, use_se=True, spatial_dropout=0.01)          # was 0.02

    # Stage 3: 12×12, 256 filters
    x = dw_sep_block(x, 256, stride=2, use_se=True, use_residual=False,
                     spatial_dropout=0.0)
    x = dw_sep_block(x, 256, use_se=True, spatial_dropout=0.025)         # was 0.05

    # Stage 4: 6×6, 512 filters
    x = dw_sep_block(x, 512, stride=2, use_se=True, use_residual=False,
                     spatial_dropout=0.0)
    x = dw_sep_block(x, 512, use_se=True, spatial_dropout=0.04)          # was 0.08
    x = dw_sep_block(x, 512, use_se=True, spatial_dropout=0.04)          # was 0.08

    # Stage 5: 3×3, 512 filters
    x = dw_sep_block(x, 512, stride=2, use_se=True, use_residual=False,
                     spatial_dropout=0.0)
    x = dw_sep_block(x, 512, use_se=True, spatial_dropout=0.04)          # was 0.08

    x = layers.GlobalAveragePooling2D()(x)

    x   = layers.Dense(256, activation='relu',
                       kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x   = layers.Dropout(0.25)(x)                                         # was 0.50
    x   = layers.Dense(128, activation='relu',
                       kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x   = layers.Dropout(0.15)(x)                                         # was 0.30
    out = layers.Dense(num_classes, activation='softmax')(x)

    return tf.keras.Model(inp, out, name='CustomDWNet_v3')


model3 = build_custom_dw_cnn(input_shape=(IMG_SIZE, IMG_SIZE, 1),
                              num_classes=len(CLASSES))
model3.summary()


def train_custom_dw_cnn(model):
    NAME    = 'CustomDWNet'
    smooth  = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05)  # was 0.10
    refine  = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.03)  # was 0.06
    cw_vals = np.array([CLASS_WEIGHTS[i] for i in range(len(CLASSES))], dtype=np.float32)

    print('\n' + '='*70)
    print('  CustomDWNet v3 | Phase 1 — Adam  peak LR=3e-4  60 ep  patience=12')
    print('='*70)
    model.compile(
        optimizer = tf.keras.optimizers.Adam(learning_rate=3e-4, clipnorm=1.0),
        loss      = smooth,
        metrics   = ['accuracy']
    )
    h1 = model.fit(
        mixed_generator_ds_gray(train_ds, cw_vals,
                                alpha=0.2, cutmix_prob=0.5),
        steps_per_epoch = STEPS_PER_EPOCH,
        validation_data = (X_val, y_val),
        epochs          = 60,
        callbacks       = make_callbacks_custom(
                              NAME + '_p1', 3e-4, 8, 60, 12,
                              phase_name='P1', min_lr=1e-6),
        verbose=1
    )

    print('\n' + '='*70)
    print('  CustomDWNet v3 | Phase 2 — Adam  peak LR=5e-5  30 ep  patience=10')
    print('='*70)
    model.compile(
        optimizer = tf.keras.optimizers.Adam(learning_rate=5e-5, clipnorm=1.0),
        loss      = refine,
        metrics   = ['accuracy']
    )
    h2 = model.fit(
        mixed_generator_ds_gray(train_ds, cw_vals,
                                alpha=0.2, cutmix_prob=0.5),
        steps_per_epoch = STEPS_PER_EPOCH,
        validation_data = (X_val, y_val),
        epochs          = 30,
        callbacks       = make_callbacks_custom(
                              NAME + '_p2', 5e-5, 3, 30, 10,
                              phase_name='P2', min_lr=1e-7),
        verbose=1
    )

    best_p1 = max(h1.history['val_accuracy'])
    best_p2 = max(h2.history['val_accuracy'])
    best    = max(best_p1, best_p2)
    print('\n' + '='*70)
    print(f'  Phase 1 best: {best_p1*100:.2f}%  |  Phase 2 best: {best_p2*100:.2f}%')
    print(f'  Overall best: {best*100:.2f}%')
    print('='*70)

    print('\n  Running TTA (6 passes)...')
    tta_preds = predict_with_tta_custom(model, X_val, n_aug=6)
    tta_acc   = np.mean(np.argmax(tta_preds, axis=1) == np.argmax(y_val, axis=1))
    print(f'  CustomDWNet TTA val accuracy: {tta_acc*100:.2f}%')

    model.save(os.path.join(SAVE_DIR, f'{NAME}_final.keras'))
    return best


acc3 = train_custom_dw_cnn(model3)

# pretrained

In [2]:
# ── Config ────────────────────────────────────────────────────────────────────
TRAIN_DIR  = '/kaggle/input/datasets/ahmedgamall/emotion-detection/Training_data/Training_data'
TEST_DIR   = '/kaggle/input/datasets/ahmedgamall/emotion-detection/test/test'
SAVE_DIR   = '/kaggle/working/models'
os.makedirs(SAVE_DIR, exist_ok=True)

IMG_SIZE   = 224    # upscaling from 96 to 224
BATCH      = 32
SEED       = 42
CLASSES    = sorted(os.listdir(TRAIN_DIR))
N_CLASSES  = len(CLASSES)

print(f'Classes ({N_CLASSES}):', CLASSES)
print(f'Image size : {IMG_SIZE}x{IMG_SIZE}x3')
print(f'Batch size : {BATCH}')

Classes (6): ['angry', 'disgust', 'fear', 'happy', 'sad', 'surprise']
Image size : 224x224x3
Batch size : 32


In [3]:
import gc
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

gc.collect()

# ═══════════════════════════════════════════════════════════════
#  DISK-BASED PIPELINE — Zero full-dataset RAM
#  Images stay on disk and load per-batch only
# ═══════════════════════════════════════════════════════════════

# ── Step 1: Collect file paths only (no images loaded) ────────
all_paths, all_labels = [], []

for label, cls in enumerate(CLASSES):
    folder = os.path.join(TRAIN_DIR, cls)
    for fname in os.listdir(folder):
        path = os.path.join(folder, fname)
        if os.path.isfile(path):
            all_paths.append(path)
            all_labels.append(label)

all_paths  = np.array(all_paths)
all_labels = np.array(all_labels, dtype=np.int32)

print('Raw distribution:')
for i, cls in enumerate(CLASSES):
    print(f'  {cls:<12} {np.sum(all_labels == i)}')

# ── Step 2: Rebalance using path indices only ─────────────────
TARGET           = 5500
balanced_indices = []

print('\nRebalancing:')
for cls_idx in range(N_CLASSES):
    cls_indices = np.where(all_labels == cls_idx)[0]
    current     = len(cls_indices)

    if current >= TARGET:
        chosen = np.random.choice(cls_indices,
                                   size    = TARGET,
                                   replace = False)
    else:
        extra  = TARGET - current
        chosen = np.concatenate([
            cls_indices,
            np.random.choice(cls_indices,
                             size    = extra,
                             replace = True)
        ])

    balanced_indices.append(chosen)
    action = 'undersampled' if current >= TARGET else 'oversampled'
    print(f'  {CLASSES[cls_idx]:<12} {current:>5} → {TARGET} ({action})')

balanced_indices = np.concatenate(balanced_indices)
rng              = np.random.default_rng(SEED)
balanced_indices = rng.permutation(balanced_indices)

bal_paths  = all_paths[balanced_indices]
bal_labels = all_labels[balanced_indices]

del all_paths, all_labels, balanced_indices
gc.collect()

print(f'\nTotal paths  : {len(bal_paths)}')
print(f'RAM used     : ~{len(bal_paths) * 200 / 1e6:.1f} MB '
      f'(strings only, no images)')

# ── Step 3: Train / val split on paths ───────────────────────
(paths_train, paths_val,
 labels_train, labels_val) = train_test_split(
    bal_paths, bal_labels,
    test_size    = 0.20,
    random_state = SEED,
    stratify     = bal_labels
)

del bal_paths, bal_labels
gc.collect()

print(f'Train paths  : {len(paths_train)}')
print(f'Val paths    : {len(paths_val)}')

# ── Step 4: Class weights ─────────────────────────────────────
CLASS_WEIGHTS = dict(enumerate(
    compute_class_weight('balanced',
                          classes = np.unique(labels_train),
                          y       = labels_train)
))
print('\nClass weights:')
for k, v in CLASS_WEIGHTS.items():
    print(f'  {CLASSES[k]:<12} {v:.4f}')

# ── Step 5: Per-image load function ──────────────────────────
def load_image(path, label):
    """
    Loads one image from disk per call.
    Called inside tf.data pipeline — never loads full dataset.
    Handles grayscale and color images automatically.
    """
    # Read raw bytes
    raw = tf.io.read_file(path)

    # Decode — force 3 channels (handles grayscale automatically)
    img = tf.image.decode_image(
        raw,
        channels         = 3,      # gray → RGB, RGBA → RGB, RGB stays RGB
        expand_animations= False
    )

    # Resize
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE],
                          method='area')

    # Cast to float32
    img = tf.cast(img, tf.float32)

    # One-hot label
    lbl = tf.one_hot(label, N_CLASSES)

    return img, lbl

# ── Step 6: Augmentation function (applied per batch) ─────────
def augment(img, lbl):
    # Horizontal flip
    img = tf.image.random_flip_left_right(img)

    # Brightness + contrast
    img = tf.image.random_brightness(img, max_delta=0.25)
    img = tf.image.random_contrast(img, lower=0.8, upper=1.2)

    # Width and height shift (up to 15%)
    pad = int(IMG_SIZE * 0.15)
    img = tf.pad(img, [[pad, pad], [pad, pad], [0, 0]],
                 mode='REFLECT')
    offset_h = tf.random.uniform([], 0, 2 * pad, dtype=tf.int32)
    offset_w = tf.random.uniform([], 0, 2 * pad, dtype=tf.int32)
    img = tf.image.crop_to_bounding_box(
        img, offset_h, offset_w, IMG_SIZE, IMG_SIZE
    )

    # Zoom (crop + resize)
    crop_size = tf.random.uniform(
        [], minval=int(IMG_SIZE * 0.80),
        maxval=IMG_SIZE, dtype=tf.int32
    )
    img = tf.image.random_crop(img, size=[crop_size, crop_size, 3])
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])

    # Clip
    img = tf.clip_by_value(img, 0.0, 255.0)
    return img, lbl

# ── Step 7: Build tf.data datasets ───────────────────────────
AUTOTUNE = tf.data.AUTOTUNE

# ── Training dataset ──────────────────────────────────────────
train_ds = (
    tf.data.Dataset
    .from_tensor_slices((paths_train, labels_train))
    .shuffle(buffer_size = 3000, seed = SEED,
             reshuffle_each_iteration = True)
    .map(load_image,  num_parallel_calls = AUTOTUNE)
    .map(augment,     num_parallel_calls = AUTOTUNE)
    .batch(BATCH)
    .prefetch(AUTOTUNE)
)

# ── Validation dataset (no augmentation) ─────────────────────
val_ds = (
    tf.data.Dataset
    .from_tensor_slices((paths_val, labels_val))
    .map(load_image,  num_parallel_calls = AUTOTUNE)
    .batch(BATCH)
    .prefetch(AUTOTUNE)
)

# ── Step 8: Build small X_val in RAM for callbacks ────────────
# EarlyStopping and ModelCheckpoint need val data
# Load ONLY val set into RAM — 20% of 33k = 6600 images
# 6600 × 96×96×3 × 4 bytes = ~0.7 GB — safe

print('\nLoading val set into RAM for callbacks...')
X_val_list, y_val_list = [], []

for path, label in zip(paths_val, labels_val):
    img = cv2.imread(path)
    if img is None:
        continue
    if len(img.shape) == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    elif img.shape[2] == 1:
        img = cv2.cvtColor(img[:,:,0], cv2.COLOR_GRAY2RGB)
    else:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE),
                     interpolation=cv2.INTER_AREA)
    X_val_list.append(img)
    y_val_list.append(label)

X_val = np.array(X_val_list, dtype=np.float32)
y_val = to_categorical(np.array(y_val_list), N_CLASSES)

del X_val_list, y_val_list
gc.collect()

print(f'X_val shape  : {X_val.shape}')
print(f'X_val RAM    : ~{X_val.nbytes / 1e9:.2f} GB')

# ── Step 9: steps_per_epoch ───────────────────────────────────
STEPS_PER_EPOCH = len(paths_train) // BATCH

print(f'\nSteps per epoch : {STEPS_PER_EPOCH}')
print(f'Val size        : {len(X_val)}')
print('\nPipeline ready — no full dataset in RAM')

# ── Verification ──────────────────────────────────────────────
print('\nVerification:')
for batch_imgs, batch_lbls in train_ds.take(1):
    print(f'  Batch shape  : {batch_imgs.shape}')
    print(f'  Batch dtype  : {batch_imgs.dtype}')
    print(f'  Pixel range  : [{batch_imgs.numpy().min():.0f},'
          f' {batch_imgs.numpy().max():.0f}]')
    print(f'  Label shape  : {batch_lbls.shape}')
    counts = batch_lbls.numpy().argmax(1)
    print(f'  Label dist   : {np.bincount(counts, minlength=N_CLASSES)}')

Raw distribution:
  angry        4500
  disgust      4500
  fear         4500
  happy        9690
  sad          5015
  surprise     5032

Rebalancing:
  angry         4500 → 5500 (oversampled)
  disgust       4500 → 5500 (oversampled)
  fear          4500 → 5500 (oversampled)
  happy         9690 → 5500 (undersampled)
  sad           5015 → 5500 (oversampled)
  surprise      5032 → 5500 (oversampled)

Total paths  : 33000
RAM used     : ~6.6 MB (strings only, no images)
Train paths  : 26400
Val paths    : 6600

Class weights:
  angry        1.0000
  disgust      1.0000
  fear         1.0000
  happy        1.0000
  sad          1.0000
  surprise     1.0000


I0000 00:00:1779017294.172006     872 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1779017294.177253     872 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5



Loading val set into RAM for callbacks...
X_val shape  : (6600, 224, 224, 3)
X_val RAM    : ~3.97 GB

Steps per epoch : 825
Val size        : 6600

Pipeline ready — no full dataset in RAM

Verification:
  Batch shape  : (32, 224, 224, 3)
  Batch dtype  : <dtype: 'float32'>
  Pixel range  : [0, 255]
  Label shape  : (32, 6)
  Label dist   : [9 5 5 7 3 3]


In [ ]:
# -- 1. Learning-rate schedule -------------------------------------------------
import albumentations as A
def make_lr_schedule(lr_max, warmup_epochs, total_epochs, min_lr=1e-6):
    """Linear warmup to lr_max, then cosine decay to min_lr."""
    def schedule(epoch, lr):
        if epoch < warmup_epochs:
            return float(lr_max * (epoch + 1) / max(warmup_epochs, 1))

        progress = (epoch - warmup_epochs) / max(total_epochs - warmup_epochs, 1)
        cosine = 0.5 * (1.0 + np.cos(np.pi * progress))

        return float(min_lr + (lr_max - min_lr) * cosine)

    return schedule


# -- 2. Callbacks factory ------------------------------------------------------
def make_callbacks(name, lr_max, warmup_epochs, total_epochs,
                   patience_es=8, min_lr=1e-6):
    ckpt_path = os.path.join(SAVE_DIR, f'{name}_best.keras')
    log_path = os.path.join(SAVE_DIR, f'{name}_log.csv')

    return [
        tf.keras.callbacks.TerminateOnNaN(),

        tf.keras.callbacks.LearningRateScheduler(
            make_lr_schedule(
                lr_max=lr_max,
                warmup_epochs=warmup_epochs,
                total_epochs=total_epochs,
                min_lr=min_lr
            ),
            verbose=1
        ),

        EarlyStopping(
            monitor='val_accuracy',
            mode='max',
            patience=patience_es,
            restore_best_weights=True,
            verbose=1
        ),

        ModelCheckpoint(
            filepath=ckpt_path,
            monitor='val_accuracy',
            mode='max',
            save_best_only=True,
            verbose=1
        ),

        CSVLogger(log_path)
    ]


# -- 3. Head builder -----------------------------------------------------------
def build_model(base, name, preprocess_fn=None,
                dense1=256, dense2=0,
                drop1=0.35, drop2=0.45,
                l2_reg=2e-4):
    inp = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='input_image')

    x = layers.Lambda(preprocess_fn, name='preprocess')(inp) if preprocess_fn else inp

    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D(name='gap')(x)

    x = layers.BatchNormalization(name='head_bn')(x)
    x = layers.Dropout(drop1, name='drop_1')(x)

    x = layers.Dense(
        dense1,
        activation='swish',
        kernel_regularizer=l2(l2_reg),
        name='dense_1'
    )(x)
    x = layers.BatchNormalization(name='dense_1_bn')(x)
    x = layers.Dropout(drop2, name='drop_2')(x)

    if dense2 and dense2 > 0:
        x = layers.Dense(
            dense2,
            activation='swish',
            kernel_regularizer=l2(l2_reg),
            name='dense_2'
        )(x)
        x = layers.BatchNormalization(name='dense_2_bn')(x)
        x = layers.Dropout(drop2, name='drop_3')(x)

    out = layers.Dense(
        N_CLASSES,
        activation='softmax',
        dtype='float32',
        name='predictions'
    )(x)

    return tf.keras.Model(inp, out, name=name)


# -- 4. Plot helper ------------------------------------------------------------
def plot_history(h1, h2, name):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    p1_len = len(h1.history['accuracy'])

    for ax, metric, title in zip(axes, ['accuracy', 'loss'], ['Accuracy', 'Loss']):
        ep1 = range(p1_len)
        ep2 = range(p1_len, p1_len + len(h2.history[metric]))

        ax.plot(ep1, h1.history[metric], label='P1 train')
        ax.plot(ep1, h1.history[f'val_{metric}'], label='P1 val', linestyle='--')
        ax.plot(ep2, h2.history[metric], label='P2 train')
        ax.plot(ep2, h2.history[f'val_{metric}'], label='P2 val', linestyle='--')

        ax.axvline(
            p1_len,
            color='gray',
            linestyle=':',
            linewidth=0.8,
            label='Phase boundary'
        )

        ax.set_title(f'{name} - {title}')
        ax.set_xlabel('Epoch')
        ax.legend(fontsize=8)

    plt.tight_layout()
    plt.show()

randaug = A.Compose([
    A.OneOf([
        A.Equalize(p=1.0),
        A.Sharpen(alpha=(0.2, 0.5), p=1.0),
        A.ColorJitter(brightness=0.2,
                      contrast=0.2,
                      saturation=0.2,
                      hue=0.05, p=1.0),
        A.GaussNoise(var_limit=(5, 20), p=1.0),
        A.Blur(blur_limit=3, p=1.0),
        A.RandomBrightnessContrast(
                      brightness_limit=0.2,
                      contrast_limit=0.2, p=1.0),
        A.RandomShadow(p=1.0),
    ], p=0.60),
    A.HorizontalFlip(p=0.5),
])
def apply_randaug_batch(X):
    out = np.empty_like(X)
    for i in range(len(X)):
        img = X[i].clip(0, 255).astype(np.uint8)  # already 0-255, no multiply
        img = randaug(image=img)['image']
        out[i] = img.astype(np.float32)            # keep as 0-255, no divide
    return out

def random_erasing_batch(X, prob=0.5, sl=0.02, sh=0.25, r1=0.3):
    out = X.copy()
    H, W = X.shape[1], X.shape[2]
    area = H * W
    for i in range(len(out)):
        if np.random.rand() > prob:
            continue
        for _ in range(10):
            erase_area = np.random.uniform(sl, sh) * area
            aspect     = np.random.uniform(r1, 1.0 / r1)
            eh = int(np.sqrt(erase_area * aspect))
            ew = int(np.sqrt(erase_area / aspect))
            if eh >= H or ew >= W:
                continue
            ey = np.random.randint(0, H - eh)
            ex = np.random.randint(0, W - ew)
            out[i, ey:ey+eh, ex:ex+ew, :] = np.mean(out[i])
            break
    return out
# ── Mixup + CutMix alternating generator ─────────────────────────────────────
def mixed_generator_ds(dataset, cw_vals, alpha=0.3, cutmix_prob=0.5):
    """
    Randomly alternates between MixUp and CutMix each batch.
    Now includes RandAugment (per image) + RandomErasing (post-mix).
    """
    ds_iter = iter(dataset.repeat())
    while True:
        X1, y1 = next(ds_iter)
        X2, y2 = next(ds_iter)
        X1, y1 = X1.numpy(), y1.numpy()
        X2, y2 = X2.numpy(), y2.numpy()
        bs     = min(len(X1), len(X2))
        X1, y1 = X1[:bs], y1[:bs]
        X2, y2 = X2[:bs], y2[:bs]

        # ── RandAugment before mixing ─────────────────────────────────────
        X1 = apply_randaug_batch(X1)
        X2 = apply_randaug_batch(X2)

        if np.random.rand() < cutmix_prob:
            # ── CutMix ───────────────────────────────────────────────────
            lam   = np.random.beta(alpha, alpha)
            H, W  = X1.shape[1], X1.shape[2]
            cut_h = int(H * np.sqrt(1.0 - lam))
            cut_w = int(W * np.sqrt(1.0 - lam))
            cx    = np.random.randint(0, W)
            cy    = np.random.randint(0, H)
            x1_c  = np.clip(cx - cut_w // 2, 0, W)
            y1_c  = np.clip(cy - cut_h // 2, 0, H)
            x2_c  = np.clip(cx + cut_w // 2, 0, W)
            y2_c  = np.clip(cy + cut_h // 2, 0, H)
            X_mix = X1.copy()
            X_mix[:, y1_c:y2_c, x1_c:x2_c, :] = X2[:, y1_c:y2_c, x1_c:x2_c, :]
            lam_eff = 1.0 - (y2_c - y1_c) * (x2_c - x1_c) / (H * W)
            y_mix   = lam_eff * y1 + (1.0 - lam_eff) * y2
        else:
            # ── MixUp ────────────────────────────────────────────────────
            lam     = np.random.beta(alpha, alpha)
            X_mix   = lam * X1 + (1 - lam) * X2
            y_mix   = lam * y1 + (1 - lam) * y2
            lam_eff = lam

        # ── RandomErasing after mixing ────────────────────────────────────
        X_mix = random_erasing_batch(X_mix, prob=0.3)

        w1    = cw_vals[y1.argmax(axis=1)]
        w2    = cw_vals[y2.argmax(axis=1)]
        w_mix = (lam_eff * w1 + (1 - lam_eff) * w2).astype(np.float32)
        yield X_mix, y_mix, w_mix
# ── Test-Time Augmentation ────────────────────────────────────────────────────
def predict_with_tta(model, X, n_aug=6):
    """
    Averages predictions over n_aug augmented versions of each image.
    Uses horizontal flip + slight brightness/zoom jitter.
    """
    tta_aug = ImageDataGenerator(
        horizontal_flip   = True,
        zoom_range        = 0.08,
        brightness_range  = [0.88, 1.12],
        fill_mode         = 'reflect'
    )
    preds = model.predict(X, verbose=0)          # original
    for _ in range(n_aug - 1):
        aug_gen = tta_aug.flow(X, batch_size=len(X), shuffle=False)
        preds  += model.predict(next(aug_gen), verbose=0)
    return preds / n_aug
print('Shared utilities loaded.')


Shared utilities loaded.


/tmp/ipykernel_872/2445432975.py:137: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(5, 20), p=1.0),


In [ ]:
def train_model2():
    NAME            = 'EfficientNetB3'
    UNFREEZE_LAYERS = 200           # ← restored from 160

    # ── Phase 1 ───────────────────────────────────────────────
    P1_EPOCHS       = 32
    P1_PEAK_LR      = 5e-4
    P1_WARMUP       = 3
    P1_PATIENCE_ES  = 15

    # ── Phase 2 ───────────────────────────────────────────────
    P2_EPOCHS       = 120
    P2_PEAK_LR      = 1e-4          # ← restored from 6e-5
    P2_WARMUP       = 5
    P2_PATIENCE_ES  = 22

    # ── Head ──────────────────────────────────────────────────
    DENSE1, DENSE2  = 512, 256
    DROP1,  DROP2   = 0.45 , 0.30
    L2_REG          = 2e-4
    LABEL_SMOOTH    = 0.08

    smooth_loss = tf.keras.losses.CategoricalCrossentropy(
        label_smoothing=LABEL_SMOOTH
    )

    cw_vals = np.array(
        [CLASS_WEIGHTS[i] for i in range(N_CLASSES)],
        dtype=np.float32
    )

    def make_callbacks_b3(tag, lr_max, warmup_epochs,
                           total_epochs, patience_es, min_lr):
        return [
            TerminateOnNaN(),
            tf.keras.callbacks.LearningRateScheduler(
                make_lr_schedule(
                    lr_max        = lr_max,
                    warmup_epochs = warmup_epochs,
                    total_epochs  = total_epochs,
                    min_lr        = min_lr
                ),
                verbose=1
            ),
            EarlyStopping(
                monitor              = 'val_accuracy',
                mode                 = 'max',
                patience             = patience_es,
                restore_best_weights = True,
                verbose              = 1
            ),
            ModelCheckpoint(
                os.path.join(SAVE_DIR, f'{tag}_best.keras'),
                monitor        = 'val_accuracy',
                mode           = 'max',
                save_best_only = True,
                verbose        = 1
            ),
            CSVLogger(os.path.join(SAVE_DIR, f'{tag}_log.csv'))
        ]

    base = EfficientNetB3(
        include_top  = False,
        weights      = 'imagenet',
        input_shape  = (IMG_SIZE, IMG_SIZE, 3)
    )
    model = build_model(
        base, NAME,
        preprocess_fn = ef_lib.preprocess_input,
        dense1        = DENSE1,
        dense2        = DENSE2,
        drop1         = DROP1,
        drop2         = DROP2,
        l2_reg        = L2_REG
    )

    # ── Phase 1 ───────────────────────────────────────────────
    print(f'\n{"="*60}')
    print(f'  {NAME} | Phase 1 — head only (base FROZEN)')
    print(f'  Peak LR={P1_PEAK_LR}  Epochs={P1_EPOCHS}  Warmup={P1_WARMUP}')
    print(f'{"="*60}')

    base.trainable = False
    model.compile(
        optimizer = AdamW(learning_rate=P1_PEAK_LR,
                          weight_decay=8e-5, clipnorm=1.0),
        loss      = smooth_loss,
        metrics   = ['accuracy']
    )

    h1 = model.fit(
        mixed_generator_ds(train_ds, cw_vals, alpha=0.3, cutmix_prob=0.5),
        steps_per_epoch = STEPS_PER_EPOCH,
        validation_data = (X_val, y_val),
        epochs          = P1_EPOCHS,
        callbacks       = make_callbacks_b3(
                              NAME + '_p1',
                              P1_PEAK_LR, P1_WARMUP, P1_EPOCHS,
                              P1_PATIENCE_ES, min_lr=5e-6
                          )
    )

    # ── Phase 2 ───────────────────────────────────────────────
    print(f'\n{"="*60}')
    print(f'  {NAME} | Phase 2 — fine tune top {UNFREEZE_LAYERS} layers')
    print(f'  Peak LR={P2_PEAK_LR}  Epochs={P2_EPOCHS}  Warmup={P2_WARMUP}')
    print(f'{"="*60}')

    base.trainable = True
    for layer in base.layers[:-UNFREEZE_LAYERS]:
        layer.trainable = False
    for layer in base.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False

    model.compile(
        optimizer = AdamW(learning_rate=P2_PEAK_LR,
                          weight_decay=6e-5, clipnorm=1.0),
        loss      = smooth_loss,
        metrics   = ['accuracy']
    )

    h2 = model.fit(
        mixed_generator_ds(train_ds, cw_vals, alpha=0.3, cutmix_prob=0.5),
        steps_per_epoch = STEPS_PER_EPOCH,
        validation_data = (X_val, y_val),
        epochs          = P2_EPOCHS,
        callbacks       = make_callbacks_b3(
                              NAME + '_p2',
                              P2_PEAK_LR, P2_WARMUP, P2_EPOCHS,
                              P2_PATIENCE_ES, min_lr=5e-7
                          )
    )

    plot_history(h1, h2, NAME)

    best_standard = max(
        max(h1.history['val_accuracy']),
        max(h2.history['val_accuracy'])
    )
    print(f'\n  {NAME} best val accuracy (no TTA): {best_standard*100:.2f}%')

    print('\n  Running TTA evaluation (6 augmented passes)...')
    tta_preds  = predict_with_tta(model, X_val, n_aug=6)
    tta_labels = np.argmax(y_val, axis=1)
    tta_acc    = np.mean(np.argmax(tta_preds, axis=1) == tta_labels)
    print(f'  {NAME} val accuracy with TTA: {tta_acc*100:.2f}%')

    model.save(os.path.join(SAVE_DIR, f'{NAME}_final.keras'))
    return model, best_standard, tta_acc


model2, acc2, acc2_tta = train_model2()

In [ ]:
def train_model3():
    NAME            = 'ResNet50V2'
    UNFREEZE_LAYERS = 120

    # ── Phase 1 ───────────────────────────────────────────────
    P1_EPOCHS       = 35
    P1_PEAK_LR      = 5e-4
    P1_WARMUP       = 3
    P1_PATIENCE_ES  = 16

    # ── Phase 2 ───────────────────────────────────────────────
    P2_EPOCHS       = 100
    P2_PEAK_LR      = 5e-5
    P2_WARMUP       = 5
    P2_PATIENCE_ES  = 25

    # ── Head ──────────────────────────────────────────────────
    DENSE1, DENSE2  = 512, 256
    DROP1,  DROP2   = 0.55, 0.38   # ← restored from 0.50, 0.35
    L2_REG          = 4e-4          # ← restored from 3e-4
    LABEL_SMOOTH    = 0.10

    smooth_loss = tf.keras.losses.CategoricalCrossentropy(
        label_smoothing=LABEL_SMOOTH
    )

    cw_vals = np.array(
        [CLASS_WEIGHTS[i] for i in range(N_CLASSES)],
        dtype=np.float32
    )

    def make_callbacks_resnet(tag, lr_max, warmup_epochs,
                               total_epochs, patience_es, min_lr):
        return [
            TerminateOnNaN(),
            tf.keras.callbacks.LearningRateScheduler(
                make_lr_schedule(
                    lr_max        = lr_max,
                    warmup_epochs = warmup_epochs,
                    total_epochs  = total_epochs,
                    min_lr        = min_lr
                ),
                verbose=1
            ),
            EarlyStopping(
                monitor              = 'val_accuracy',
                mode                 = 'max',
                patience             = patience_es,
                restore_best_weights = True,
                verbose              = 1
            ),
            ModelCheckpoint(
                os.path.join(SAVE_DIR, f'{tag}_best.keras'),
                monitor        = 'val_accuracy',
                mode           = 'max',
                save_best_only = True,
                verbose        = 1
            ),
            CSVLogger(os.path.join(SAVE_DIR, f'{tag}_log.csv'))
        ]

    base = ResNet50V2(
        include_top  = False,
        weights      = 'imagenet',
        input_shape  = (IMG_SIZE, IMG_SIZE, 3)
    )
    model = build_model(
        base, NAME,
        preprocess_fn = rv2_lib.preprocess_input,
        dense1        = DENSE1,
        dense2        = DENSE2,
        drop1         = DROP1,
        drop2         = DROP2,
        l2_reg        = L2_REG
    )

    # ── Phase 1 ───────────────────────────────────────────────
    print(f'\n{"="*60}')
    print(f'  {NAME} | Phase 1 — head only (base FROZEN)')
    print(f'  Peak LR={P1_PEAK_LR}  Epochs={P1_EPOCHS}  Warmup={P1_WARMUP}')
    print(f'{"="*60}')

    base.trainable = False
    model.compile(
        optimizer = AdamW(learning_rate=P1_PEAK_LR,
                          weight_decay=8e-5, clipnorm=1.0),
        loss      = smooth_loss,
        metrics   = ['accuracy']
    )

    h1 = model.fit(
        mixed_generator_ds(train_ds, cw_vals, alpha=0.3, cutmix_prob=0.5),
        steps_per_epoch = STEPS_PER_EPOCH,
        validation_data = (X_val, y_val),
        epochs          = P1_EPOCHS,
        callbacks       = make_callbacks_resnet(
                              NAME + '_p1',
                              P1_PEAK_LR, P1_WARMUP, P1_EPOCHS,
                              P1_PATIENCE_ES, min_lr=5e-6
                          )
    )

    # ── Phase 2 ───────────────────────────────────────────────
    print(f'\n{"="*60}')
    print(f'  {NAME} | Phase 2 — fine tune top {UNFREEZE_LAYERS} layers')
    print(f'  Peak LR={P2_PEAK_LR}  Epochs={P2_EPOCHS}  Warmup={P2_WARMUP}')
    print(f'{"="*60}')

    base.trainable = True
    for layer in base.layers[:-UNFREEZE_LAYERS]:
        layer.trainable = False
    for layer in base.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False

    model.compile(
        optimizer = AdamW(learning_rate=P2_PEAK_LR,
                          weight_decay=6e-5, clipnorm=1.0),
        loss      = smooth_loss,
        metrics   = ['accuracy']
    )

    h2 = model.fit(
        mixed_generator_ds(train_ds, cw_vals, alpha=0.3, cutmix_prob=0.5),
        steps_per_epoch = STEPS_PER_EPOCH,
        validation_data = (X_val, y_val),
        epochs          = P2_EPOCHS,
        callbacks       = make_callbacks_resnet(
                              NAME + '_p2',
                              P2_PEAK_LR, P2_WARMUP, P2_EPOCHS,
                              P2_PATIENCE_ES, min_lr=5e-7
                          )
    )

    plot_history(h1, h2, NAME)

    best_standard = max(
        max(h1.history['val_accuracy']),
        max(h2.history['val_accuracy'])
    )
    print(f'\n  {NAME} best val accuracy (no TTA): {best_standard*100:.2f}%')

    print('\n  Running TTA evaluation (6 augmented passes)...')
    tta_preds  = predict_with_tta(model, X_val, n_aug=6)
    tta_labels = np.argmax(y_val, axis=1)
    tta_acc    = np.mean(np.argmax(tta_preds, axis=1) == tta_labels)
    print(f'  {NAME} val accuracy with TTA: {tta_acc*100:.2f}%')

    model.save(os.path.join(SAVE_DIR, f'{NAME}_final.keras'))
    return model, best_standard, tta_acc


model3, acc3, acc3_tta = train_model3()


  ResNet50V2 | Phase 1 — head only (base FROZEN)
  Peak LR=0.0005  Epochs=35  Warmup=3

Epoch 1: LearningRateScheduler setting learning rate to 0.00016666666666666666.
Epoch 1/35


I0000 00:00:1779017321.803829     920 service.cc:152] XLA service 0x787444009660 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779017321.803870     920 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1779017321.803874     920 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1779017324.121777     920 cuda_dnn.cc:529] Loaded cuDNN version 91002


  2/825 ━━━━━━━━━━━━━━━━━━━━ 46s 57ms/step - accuracy: 0.2422 - loss: 3.3046   

I0000 00:00:1779017334.237739     920 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


825/825 ━━━━━━━━━━━━━━━━━━━━ 0s 131ms/step - accuracy: 0.2381 - loss: 2.8529
Epoch 1: val_accuracy improved from -inf to 0.48939, saving model to /kaggle/working/models/ResNet50V2_p1_best.keras
825/825 ━━━━━━━━━━━━━━━━━━━━ 158s 164ms/step - accuracy: 0.2381 - loss: 2.8526 - val_accuracy: 0.4894 - val_loss: 1.8901 - learning_rate: 1.6667e-04

Epoch 2: LearningRateScheduler setting learning rate to 0.0003333333333333333.
Epoch 2/35
825/825 ━━━━━━━━━━━━━━━━━━━━ 0s 129ms/step - accuracy: 0.3082 - loss: 2.3564
Epoch 2: val_accuracy improved from 0.48939 to 0.51833, saving model to /kaggle/working/models/ResNet50V2_p1_best.keras
825/825 ━━━━━━━━━━━━━━━━━━━━ 114s 138ms/step - accuracy: 0.3082 - loss: 2.3563 - val_accuracy: 0.5183 - val_loss: 1.8157 - learning_rate: 3.3333e-04

Epoch 3: LearningRateScheduler setting learning rate to 0.0005.
Epoch 3/35
825/825 ━━━━━━━━━━━━━━━━━━━━ 0s 129ms/step - accuracy: 0.3485 - loss: 2.1292
Epoch 3: val_accuracy improved from 0.51833 to 0.52394, saving mode

In [ ]:
from tensorflow.keras.applications.xception import preprocess_input as xcp_lib

def train_model7():
    NAME            = 'Xception'
    UNFREEZE_LAYERS = 126           # was 100 — Xception has ~134 layers, unfreeze most of them

    # ── Phase 1 ───────────────────────────────────────────────
    P1_EPOCHS       = 32
    P1_PEAK_LR      = 4e-4
    P1_WARMUP       = 4
    P1_PATIENCE_ES  = 16

    # ── Phase 2 ───────────────────────────────────────────────
    P2_EPOCHS       = 120            # was 65 — val was still climbing at ep 46, more room
    P2_PEAK_LR      = 3e-5
    P2_WARMUP       = 7
    P2_PATIENCE_ES  = 28            # was 20

    # ── Head: stronger regularization to close the ~9% train-val gap ─────────
    DENSE1, DENSE2  = 512, 256
    DROP1,  DROP2   = 0.50, 0.35   # was 0.30/0.20 — major tighten, gap was huge
    L2_REG          = 3e-4          # was 1.5e-4 — tripled
    LABEL_SMOOTH    = 0.08

    smooth_loss = tf.keras.losses.CategoricalCrossentropy(
        label_smoothing=LABEL_SMOOTH
    )

    cw_vals = np.array(
        [CLASS_WEIGHTS[i] for i in range(N_CLASSES)],
        dtype=np.float32
    )

    def make_callbacks_xception(tag, lr_max, warmup_epochs,
                                 total_epochs, patience_es, min_lr):
        return [
            TerminateOnNaN(),
            tf.keras.callbacks.LearningRateScheduler(
                make_lr_schedule(
                    lr_max        = lr_max,
                    warmup_epochs = warmup_epochs,
                    total_epochs  = total_epochs,
                    min_lr        = min_lr
                ),
                verbose=1
            ),
            EarlyStopping(
                monitor              = 'val_accuracy',
                mode                 = 'max',
                patience             = patience_es,
                restore_best_weights = True,
                verbose              = 1
            ),
            ModelCheckpoint(
                os.path.join(SAVE_DIR, f'{tag}_best.keras'),
                monitor        = 'val_accuracy',
                mode           = 'max',
                save_best_only = True,
                verbose        = 1
            ),
            CSVLogger(os.path.join(SAVE_DIR, f'{tag}_log.csv'))
        ]

    base = Xception(
        include_top  = False,
        weights      = 'imagenet',
        input_shape  = (IMG_SIZE, IMG_SIZE, 3)
    )

    model = build_model(
        base, NAME,
        preprocess_fn = xcp_lib,
        dense1        = DENSE1,
        dense2        = DENSE2,
        drop1         = DROP1,
        drop2         = DROP2,
        l2_reg        = L2_REG
    )

    # ── Phase 1: head only ────────────────────────────────────
    print(f'\n{"="*60}')
    print(f'  {NAME} | Phase 1 — head only (base FROZEN)')
    print(f'  Peak LR={P1_PEAK_LR}  Epochs={P1_EPOCHS}  Warmup={P1_WARMUP}')
    print(f'{"="*60}')

    base.trainable = False
    model.compile(
        optimizer = AdamW(
                        learning_rate = P1_PEAK_LR,
                        weight_decay  = 8e-5,
                        clipnorm      = 1.0
                    ),
        loss    = smooth_loss,
        metrics = ['accuracy']
    )

    h1 = model.fit(
        mixed_generator_ds(train_ds, cw_vals, alpha=0.3, cutmix_prob=0.5),  # CutMix added
        steps_per_epoch = STEPS_PER_EPOCH,
        validation_data = (X_val, y_val),
        epochs          = P1_EPOCHS,
        callbacks       = make_callbacks_xception(
                              NAME + '_p1',
                              P1_PEAK_LR, P1_WARMUP, P1_EPOCHS,
                              P1_PATIENCE_ES, min_lr=8e-6
                          )
    )

    # ── Phase 2: fine-tune top layers ─────────────────────────
    print(f'\n{"="*60}')
    print(f'  {NAME} | Phase 2 — top {UNFREEZE_LAYERS} layers UNFROZEN')
    print(f'  Peak LR={P2_PEAK_LR}  Epochs={P2_EPOCHS}  Warmup={P2_WARMUP}')
    print(f'{"="*60}')

    base.trainable = True
    for layer in base.layers[:-UNFREEZE_LAYERS]:
        layer.trainable = False
    for layer in base.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False

    model.compile(
        optimizer = AdamW(
                        learning_rate = P2_PEAK_LR,
                        weight_decay  = 5e-5,
                        clipnorm      = 1.0
                    ),
        loss    = smooth_loss,
        metrics = ['accuracy']
    )

    h2 = model.fit(
        mixed_generator_ds(train_ds, cw_vals, alpha=0.3, cutmix_prob=0.5),  # CutMix added
        steps_per_epoch = STEPS_PER_EPOCH,
        validation_data = (X_val, y_val),
        epochs          = P2_EPOCHS,
        callbacks       = make_callbacks_xception(
                              NAME + '_p2',
                              P2_PEAK_LR, P2_WARMUP, P2_EPOCHS,
                              P2_PATIENCE_ES, min_lr=1e-7
                          )
    )

    plot_history(h1, h2, NAME)

    best_standard = max(
        max(h1.history['val_accuracy']),
        max(h2.history['val_accuracy'])
    )
    print(f'\n  {NAME} best val accuracy (no TTA): {best_standard*100:.2f}%')

    # ── TTA evaluation ────────────────────────────────────────
    print('\n  Running TTA evaluation (6 augmented passes)...')
    tta_preds  = predict_with_tta(model, X_val, n_aug=6)
    tta_labels = np.argmax(y_val, axis=1)
    tta_acc    = np.mean(np.argmax(tta_preds, axis=1) == tta_labels)
    print(f'  {NAME} val accuracy with TTA: {tta_acc*100:.2f}%')

    model.save(os.path.join(SAVE_DIR, f'{NAME}_final.keras'))
    return model, best_standard, tta_acc


model7, acc7, acc7_tta = train_model7()

In [ ]:
def mixed_generator_ds(dataset, cw_vals, alpha=0.3, cutmix_prob=0.5):
    """Randomly alternates between MixUp and CutMix each batch."""
    ds_iter = iter(dataset.repeat())
    while True:
        X1, y1 = next(ds_iter)
        X2, y2 = next(ds_iter)
        X1, y1 = X1.numpy(), y1.numpy()
        X2, y2 = X2.numpy(), y2.numpy()

        bs     = min(len(X1), len(X2))
        X1, y1 = X1[:bs], y1[:bs]
        X2, y2 = X2[:bs], y2[:bs]

        # ── RandAugment before mixing ─────────────────────────
        X1 = apply_randaug_batch(X1)
        X2 = apply_randaug_batch(X2)

        if np.random.rand() < cutmix_prob:
            # ── CutMix ───────────────────────────────────────────────────────
            lam   = np.random.beta(alpha, alpha)
            H, W  = X1.shape[1], X1.shape[2]
            cut_h = int(H * np.sqrt(1.0 - lam))
            cut_w = int(W * np.sqrt(1.0 - lam))
            cx    = np.random.randint(0, W)
            cy    = np.random.randint(0, H)
            x1_c  = np.clip(cx - cut_w // 2, 0, W)
            y1_c  = np.clip(cy - cut_h // 2, 0, H)
            x2_c  = np.clip(cx + cut_w // 2, 0, W)
            y2_c  = np.clip(cy + cut_h // 2, 0, H)

            X_mix = X1.copy()
            X_mix[:, y1_c:y2_c, x1_c:x2_c, :] = X2[:, y1_c:y2_c, x1_c:x2_c, :]
            lam_eff = 1.0 - (y2_c - y1_c) * (x2_c - x1_c) / (H * W)
            y_mix   = lam_eff * y1 + (1.0 - lam_eff) * y2
        else:
            # ── MixUp ────────────────────────────────────────────────────────
            lam   = np.random.beta(alpha, alpha)
            X_mix = lam * X1 + (1 - lam) * X2
            y_mix = lam * y1 + (1 - lam) * y2
            lam_eff = lam

        # ── RandomErasing after mixing ────────────────────────
        X_mix = random_erasing_batch(X_mix, prob=0.3)

        w1    = cw_vals[y1.argmax(axis=1)]
        w2    = cw_vals[y2.argmax(axis=1)]
        w_mix = (lam_eff * w1 + (1 - lam_eff) * w2).astype(np.float32)

        yield X_mix, y_mix, w_mix


def train_model4():
    NAME            = 'ConvNeXtSmall'
    UNFREEZE_LAYERS = 170

    # ── Phase 1 ───────────────────────────────────────────────
    P1_EPOCHS       = 35
    P1_PEAK_LR      = 3e-4
    P1_WARMUP       = 5
    P1_PATIENCE_ES  = 16

    # ── Phase 2 ───────────────────────────────────────────────
    P2_EPOCHS       = 120          # was 95
    P2_PEAK_LR      = 5e-5
    P2_WARMUP       = 7
    P2_PATIENCE_ES  = 28

    # ── Head ──────────────────────────────────────────────────
    DENSE1, DENSE2  = 512, 256
    DROP1,  DROP2   = 0.50, 0.35   # was 0.55/0.38
    L2_REG          = 3e-4          # was 4e-4
    LABEL_SMOOTH    = 0.08

    smooth_loss = tf.keras.losses.CategoricalCrossentropy(
        label_smoothing=LABEL_SMOOTH
    )

    cw_vals = np.array(
        [CLASS_WEIGHTS[i] for i in range(N_CLASSES)],
        dtype=np.float32
    )

    def make_callbacks_convnext(tag, lr_max, warmup_epochs,
                                 total_epochs, patience_es, min_lr):
        return [
            TerminateOnNaN(),
            tf.keras.callbacks.LearningRateScheduler(
                make_lr_schedule(
                    lr_max        = lr_max,
                    warmup_epochs = warmup_epochs,
                    total_epochs  = total_epochs,
                    min_lr        = min_lr
                ),
                verbose=1
            ),
            EarlyStopping(
                monitor              = 'val_accuracy',
                mode                 = 'max',
                patience             = patience_es,
                restore_best_weights = True,
                verbose              = 1
            ),
            ModelCheckpoint(
                os.path.join(SAVE_DIR, f'{tag}_best.keras'),
                monitor        = 'val_accuracy',
                mode           = 'max',
                save_best_only = True,
                verbose        = 1
            ),
            CSVLogger(os.path.join(SAVE_DIR, f'{tag}_log.csv'))
        ]

    base = tf.keras.applications.ConvNeXtSmall(
        include_top   = False,
        weights       = 'imagenet',
        input_shape   = (IMG_SIZE, IMG_SIZE, 3)
    )

    inputs  = base.input
    x       = base.output
    x       = tf.keras.layers.GlobalAveragePooling2D()(x)
    x       = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x)
    x       = tf.keras.layers.Dense(
                  DENSE1,
                  kernel_regularizer=tf.keras.regularizers.l2(L2_REG)
              )(x)
    x       = tf.keras.layers.Activation('gelu')(x)
    x       = tf.keras.layers.Dropout(DROP1)(x)
    x       = tf.keras.layers.Dense(
                  DENSE2,
                  kernel_regularizer=tf.keras.regularizers.l2(L2_REG)
              )(x)
    x       = tf.keras.layers.Activation('gelu')(x)
    x       = tf.keras.layers.Dropout(DROP2)(x)
    outputs = tf.keras.layers.Dense(N_CLASSES, activation='softmax')(x)
    model   = tf.keras.Model(inputs, outputs, name=NAME)

    print(f'\n  {NAME}: {model.count_params():,} total parameters')

    # ── Phase 1 ───────────────────────────────────────────────
    print(f'\n{"="*60}')
    print(f'  {NAME} | Phase 1 — head only (base FROZEN)')
    print(f'  Peak LR={P1_PEAK_LR}  Epochs={P1_EPOCHS}  Warmup={P1_WARMUP}')
    print(f'{"="*60}')

    base.trainable = False
    model.compile(
        optimizer = AdamW(learning_rate=P1_PEAK_LR,
                          weight_decay=8e-5, clipnorm=1.0),
        loss      = smooth_loss,
        metrics   = ['accuracy']
    )

    h1 = model.fit(
        mixed_generator_ds(train_ds, cw_vals, alpha=0.3, cutmix_prob=0.5),
        steps_per_epoch = STEPS_PER_EPOCH,
        validation_data = (X_val, y_val),
        epochs          = P1_EPOCHS,
        callbacks       = make_callbacks_convnext(
                              NAME + '_p1',
                              P1_PEAK_LR, P1_WARMUP, P1_EPOCHS,
                              P1_PATIENCE_ES, min_lr=1e-6
                          )
    )

    # ── Phase 2 ───────────────────────────────────────────────
    print(f'\n{"="*60}')
    print(f'  {NAME} | Phase 2 — fine tune top {UNFREEZE_LAYERS} layers')
    print(f'  Peak LR={P2_PEAK_LR}  Epochs={P2_EPOCHS}  Warmup={P2_WARMUP}')
    print(f'{"="*60}')

    base.trainable = True
    for layer in base.layers[:-UNFREEZE_LAYERS]:
        layer.trainable = False
    # Do NOT freeze LayerNorm — intentionally omitting the BN freeze step

    model.compile(
        optimizer = AdamW(learning_rate=P2_PEAK_LR,
                          weight_decay=6e-5, clipnorm=1.0),
        loss      = smooth_loss,
        metrics   = ['accuracy']
    )

    h2 = model.fit(
        mixed_generator_ds(train_ds, cw_vals, alpha=0.3, cutmix_prob=0.5),
        steps_per_epoch = STEPS_PER_EPOCH,
        validation_data = (X_val, y_val),
        epochs          = P2_EPOCHS,
        callbacks       = make_callbacks_convnext(
                              NAME + '_p2',
                              P2_PEAK_LR, P2_WARMUP, P2_EPOCHS,
                              P2_PATIENCE_ES, min_lr=5e-7
                          )
    )

    plot_history(h1, h2, NAME)

    best_standard = max(
        max(h1.history['val_accuracy']),
        max(h2.history['val_accuracy'])
    )
    print(f'\n  {NAME} best val accuracy (no TTA): {best_standard*100:.2f}%')

    print('\n  Running TTA evaluation (6 augmented passes)...')
    tta_preds  = predict_with_tta(model, X_val, n_aug=6)
    tta_labels = np.argmax(y_val, axis=1)
    tta_acc    = np.mean(np.argmax(tta_preds, axis=1) == tta_labels)
    print(f'  {NAME} val accuracy with TTA: {tta_acc*100:.2f}%')

    model.save(os.path.join(SAVE_DIR, f'{NAME}_final.keras'))
    return model, best_standard, tta_acc


model4, acc4, acc4_tta = train_model4()

# Transformer

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint, CSVLogger, TerminateOnNaN
)
import numpy as np
import os
 
# ══════════════════════════════════════════════════════════════════════════════
# Shared: Stochastic Depth
# ══════════════════════════════════════════════════════════════════════════════
class StochasticDepth(tf.keras.layers.Layer):
    """
    Drop entire residual branches during training (linearly scaled per layer).
    Critical for ViT stability — without it, deep transformer blocks
    co-adapt and become brittle to LR changes.
    """
    def __init__(self, drop_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.drop_rate = drop_rate
 
    def call(self, x, training=None):
        if not training or self.drop_rate == 0.0:
            return x
        keep  = 1.0 - self.drop_rate
        shape = (tf.shape(x)[0],) + (1,) * (len(x.shape) - 1)
        rand  = tf.floor(keep + tf.random.uniform(shape, dtype=x.dtype))
        return (x / keep) * rand
 
    def get_config(self):
        cfg = super().get_config()
        cfg.update({'drop_rate': self.drop_rate})
        return cfg
 
 

In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# MODEL 1 — CustomViT v3
# ══════════════════════════════════════════════════════════════════════════════
 
def _vit_transformer_block(x, d_model, num_heads, mlp_dim,
                            attn_drop=0.0, ffn_drop=0.1, path_drop=0.0):
    """
    Pre-norm transformer block.
    attn_drop=0.0 intentionally — attention dropout on scratch ViTs
    destabilises attention maps before they converge, causing val spikes.
    """
    # Self-attention branch
    norm1 = layers.LayerNormalization(epsilon=1e-6)(x)
    attn  = layers.MultiHeadAttention(
        num_heads = num_heads,
        key_dim   = d_model // num_heads,
        dropout   = attn_drop       # kept at 0.0
    )(norm1, norm1)
    attn = StochasticDepth(drop_rate=path_drop)(attn)
    x    = layers.Add()([x, attn])
 
    # FFN branch
    norm2 = layers.LayerNormalization(epsilon=1e-6)(x)
    ff    = layers.Dense(mlp_dim, activation='gelu')(norm2)
    ff    = layers.Dropout(ffn_drop)(ff)
    ff    = layers.Dense(d_model)(ff)
    ff    = layers.Dropout(ffn_drop)(ff)
    ff    = StochasticDepth(drop_rate=path_drop)(ff)
    x     = layers.Add()([x, ff])
    return x
 
 
def _vit_conv_stem(x, d_model):
    """
    4-stage residual conv stem.
    With 224×224 input: 224 → 112 → 56 → 28 → 14
    Output: (B, 14, 14, d_model) → 196 tokens
    """
    # Stage 1: 224 → 112
    x = layers.Conv2D(32, 3, strides=2, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('gelu')(x)
 
    # Stage 2: 112 → 56 (with residual)
    shortcut = layers.Conv2D(64, 1, strides=2, padding='same', use_bias=False)(x)
    x = layers.Conv2D(64, 3, strides=2, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('gelu')(x)
    x = layers.Conv2D(64, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, shortcut])
    x = layers.Activation('gelu')(x)
 
    # Stage 3: 56 → 28 (with residual)
    shortcut = layers.Conv2D(128, 1, strides=2, padding='same', use_bias=False)(x)
    x = layers.Conv2D(128, 3, strides=2, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('gelu')(x)
    x = layers.Conv2D(128, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, shortcut])
    x = layers.Activation('gelu')(x)
 
    # Stage 4: 28 → 14 — project to d_model
    x = layers.Conv2D(d_model, 3, strides=2, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('gelu')(x)
 
    return x   # (B, 14, 14, d_model) → 196 tokens
 
 
def build_vit_v3(input_shape=(224, 224, 3), num_classes=6,
                 d_model=192, num_heads=6, mlp_dim=384,
                 num_layers=8, ffn_drop=0.12):
    """
    CustomViT v3 — adapted for RGB 224×224 pipeline.
 
    Changes from original:
      - Input: (96,96,1) → (224,224,3)
      - Rescaling(1/255) built into model (pipeline sends 0–255)
      - Conv stem now outputs 14×14 = 196 tokens (was 6×6 = 36)
      - num_patches = (224//16)^2 = 196
      - Everything else unchanged (attn_drop=0, 8 layers, d_model=192)
    """
    H, W, _ = input_shape
    num_patches = (H // 16) * (W // 16)   # 14×14 = 196
 
    inp = layers.Input(shape=input_shape)
 
    # Normalize 0–255 → 0–1  (pipeline keeps raw pixel values)
    x = layers.Rescaling(1. / 255)(inp)
 
    # Conv stem → patch tokens
    x = _vit_conv_stem(x, d_model)              # (B, 14, 14, d_model)
    x = layers.Reshape((num_patches, d_model))(x)  # (B, 196, d_model)
 
    # Positional embeddings
    positions = tf.range(start=0, limit=num_patches)
    pos_emb   = layers.Embedding(input_dim=num_patches,
                                  output_dim=d_model)(positions)
    x = x + pos_emb
    x = layers.Dropout(0.10)(x)
 
    # Transformer blocks with linearly increasing stochastic depth
    for i in range(num_layers):
        path_drop = 0.15 * i / max(num_layers - 1, 1)   # 0.0 → 0.15
        x = _vit_transformer_block(
            x,
            d_model   = d_model,
            num_heads = num_heads,
            mlp_dim   = mlp_dim,
            attn_drop = 0.0,
            ffn_drop  = ffn_drop,
            path_drop = path_drop
        )
 
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    x = layers.GlobalAveragePooling1D()(x)
 
    # Classifier head
    x   = layers.Dense(256, activation='gelu',
                        kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x   = layers.Dropout(0.40)(x)
    x   = layers.Dense(128, activation='gelu',
                        kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x   = layers.Dropout(0.25)(x)
    out = layers.Dense(num_classes, activation='softmax', dtype='float32')(x)
 
    return tf.keras.Model(inp, out, name='CustomViT_v3')
 
 
def train_vit_v3(model):
    """
    Training adapted to use mixed_generator_ds + STEPS_PER_EPOCH.
 
    Phase 1: Adam  smooth=0.08  peak LR=1.5e-4  80 ep  patience=15
    Phase 2: Adam  smooth=0.04  peak LR=8e-5    40 ep  patience=12
    """
    NAME    = 'CustomViT_v3'
    smooth  = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.08)
    refine  = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.04)
    cw_vals = np.array([CLASS_WEIGHTS[i] for i in range(N_CLASSES)],
                        dtype=np.float32)
 
    def _cbs(tag, lr_max, warmup, total, patience, min_lr=1e-7):
        return [
            TerminateOnNaN(),
            tf.keras.callbacks.LearningRateScheduler(
                make_lr_schedule(lr_max, warmup, total, min_lr), verbose=1),
            EarlyStopping(monitor='val_accuracy', mode='max',
                          patience=patience, restore_best_weights=True,
                          verbose=1),
            ModelCheckpoint(
                os.path.join(SAVE_DIR, f'{tag}_best.keras'),
                monitor='val_accuracy', mode='max',
                save_best_only=True, verbose=1),
            CSVLogger(os.path.join(SAVE_DIR, f'{tag}_log.csv')),
        ]
 
    # ── Phase 1 ───────────────────────────────────────────────
    print(f'\n{"="*60}')
    print(f'  {NAME} | Phase 1 — Adam  peak LR=1.5e-4  80 ep  patience=15')
    print(f'{"="*60}')
    model.compile(
        optimizer = tf.keras.optimizers.Adam(learning_rate=1.5e-4, clipnorm=1.0),
        loss      = smooth,
        metrics   = ['accuracy']
    )
    h1 = model.fit(
        mixed_generator_ds(train_ds, cw_vals, alpha=0.3, cutmix_prob=0.5),
        steps_per_epoch = STEPS_PER_EPOCH,
        validation_data = (X_val, y_val),
        epochs          = 80,
        callbacks       = _cbs(NAME + '_p1', 1.5e-4, 10, 80, 15, min_lr=1e-6)
    )
 
    # ── Phase 2 ───────────────────────────────────────────────
    print(f'\n{"="*60}')
    print(f'  {NAME} | Phase 2 — Adam  peak LR=8e-5   40 ep  patience=12')
    print(f'{"="*60}')
    model.compile(
        optimizer = tf.keras.optimizers.Adam(learning_rate=8e-5, clipnorm=1.0),
        loss      = refine,
        metrics   = ['accuracy']
    )
    h2 = model.fit(
        mixed_generator_ds(train_ds, cw_vals, alpha=0.3, cutmix_prob=0.5),
        steps_per_epoch = STEPS_PER_EPOCH,
        validation_data = (X_val, y_val),
        epochs          = 40,
        callbacks       = _cbs(NAME + '_p2', 8e-5, 2, 40, 12, min_lr=1e-7)
    )
 
    plot_history(h1, h2, NAME)
 
    best_p1 = max(h1.history['val_accuracy'])
    best_p2 = max(h2.history['val_accuracy'])
    best    = max(best_p1, best_p2)
    print(f'\n  Phase 1 best: {best_p1*100:.2f}%  |  Phase 2 best: {best_p2*100:.2f}%')
    print(f'  Overall best: {best*100:.2f}%')
 
    print('\n  Running TTA (6 passes)...')
    tta_preds = predict_with_tta(model, X_val, n_aug=6)
    tta_acc   = np.mean(np.argmax(tta_preds, axis=1) == np.argmax(y_val, axis=1))
    print(f'  {NAME} TTA val accuracy: {tta_acc*100:.2f}%')
 
    model.save(os.path.join(SAVE_DIR, f'{NAME}_final.keras'))
    return model, best, tta_acc
 
 
# ── Build & train ViT v3 ──────────────────────────────────────
model_vit = build_vit_v3(input_shape=(IMG_SIZE, IMG_SIZE, 3),
                          num_classes=N_CLASSES)
model_vit.summary()
model_vit, acc_vit, acc_vit_tta = train_vit_v3(model_vit)
 
 

In [ ]:
class AddPositionEmbedding(layers.Layer):
    """Learnable positional embeddings added to patch tokens."""
    def __init__(self, n_patches, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.n_patches   = n_patches
        self.embed_dim   = embed_dim
        self.pos_embedding = layers.Embedding(input_dim=n_patches,
                                               output_dim=embed_dim)
 
    def call(self, x):
        positions = tf.range(start=0, limit=self.n_patches, delta=1)
        pos = self.pos_embedding(positions)   # (n_patches, embed_dim)
        pos = tf.expand_dims(pos, axis=0)     # (1, n_patches, embed_dim)
        return x + pos
 
    def get_config(self):
        cfg = super().get_config()
        cfg.update({'n_patches': self.n_patches, 'embed_dim': self.embed_dim})
        return cfg
 
 
def train_model_hybrid():
    NAME = 'HybridCNNViT'
 
    # ── Hyperparameters ───────────────────────────────────────
    P1_EPOCHS      = 40
    P1_PEAK_LR     = 3e-4
    P1_WARMUP      = 8
    P1_PATIENCE_ES = 18
 
    P2_EPOCHS      = 60
    P2_PEAK_LR     = 6e-5
    P2_WARMUP      = 5
    P2_PATIENCE_ES = 20
 
    DROP_ATT     = 0.10   # kept low — scratch ViT attention needs stability
    DROP_HEAD    = 0.45
    L2_REG       = 1e-4
    LABEL_SMOOTH = 0.10
 
    smooth_loss = tf.keras.losses.CategoricalCrossentropy(
        label_smoothing=LABEL_SMOOTH
    )
    cw_vals = np.array([CLASS_WEIGHTS[i] for i in range(N_CLASSES)],
                        dtype=np.float32)
 
    # ── Architecture helpers ───────────────────────────────────
    def conv_bn_gelu(x, filters, kernel=3, strides=1, name=''):
        x = layers.Conv2D(filters, kernel, strides=strides, padding='same',
                           use_bias=False, name=f'{name}_conv')(x)
        x = layers.BatchNormalization(name=f'{name}_bn')(x)
        x = layers.Activation('gelu', name=f'{name}_gelu')(x)
        return x
 
    def depthwise_conv_block(x, filters, strides=1, name=''):
        x = layers.DepthwiseConv2D(3, strides=strides, padding='same',
                                    use_bias=False, name=f'{name}_dw')(x)
        x = layers.BatchNormalization(name=f'{name}_dw_bn')(x)
        x = layers.Activation('gelu', name=f'{name}_dw_gelu')(x)
        x = layers.Conv2D(filters, 1, use_bias=False, name=f'{name}_pw')(x)
        x = layers.BatchNormalization(name=f'{name}_pw_bn')(x)
        x = layers.Activation('gelu', name=f'{name}_pw_gelu')(x)
        return x
 
    def hybrid_transformer_block(x, num_heads, ff_dim, dropout=0.1, name=''):
        """Pre-norm transformer block for the hybrid model."""
        x_norm = layers.LayerNormalization(epsilon=1e-6,
                                            name=f'{name}_ln1')(x)
        attn   = layers.MultiHeadAttention(
                     num_heads = num_heads,
                     key_dim   = x.shape[-1] // num_heads,
                     dropout   = dropout,
                     name      = f'{name}_mha'
                 )(x_norm, x_norm)
        # No extra Dropout after attn — dropout is already inside MHA
        x      = layers.Add(name=f'{name}_add1')([x, attn])
 
        x_norm = layers.LayerNormalization(epsilon=1e-6,
                                            name=f'{name}_ln2')(x)
        ff     = layers.Dense(ff_dim, activation='gelu',
                               name=f'{name}_ff1')(x_norm)
        ff     = layers.Dropout(dropout, name=f'{name}_ff_drop')(ff)
        ff     = layers.Dense(x.shape[-1], name=f'{name}_ff2')(ff)
        x      = layers.Add(name=f'{name}_add2')([x, ff])
        return x
 
    # ── Build model ───────────────────────────────────────────
    reg = tf.keras.regularizers.l2(L2_REG)
    inp = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='input_image')
 
    # Normalize 0–255 → 0–1  (pipeline sends raw float32 pixels)
    x = layers.Rescaling(1. / 255, name='normalize')(inp)
 
    # CNN backbone: hierarchical local feature extraction
    # With IMG_SIZE=224: 224→112→56→28→14
    x = conv_bn_gelu(x, 32, strides=2, name='s1_conv1')    # 112×112×32
    x = conv_bn_gelu(x, 32,            name='s1_conv2')    # 112×112×32
 
    x = depthwise_conv_block(x, 64,  strides=2, name='s2_dw1')  # 56×56×64
    x = depthwise_conv_block(x, 64,             name='s2_dw2')  # 56×56×64
 
    x = depthwise_conv_block(x, 128, strides=2, name='s3_dw1')  # 28×28×128
    x = depthwise_conv_block(x, 128,            name='s3_dw2')  # 28×28×128
 
    x = depthwise_conv_block(x, 256, strides=2, name='s4_dw1')  # 14×14×256
    x = depthwise_conv_block(x, 256,            name='s4_dw2')  # 14×14×256
 
    # Bridge: spatial grid → patch sequence
    h_feat    = x.shape[1]          # 14
    w_feat    = x.shape[2]          # 14
    c_feat    = x.shape[3]          # 256
    n_patches = h_feat * w_feat     # 196
 
    x = layers.Reshape((n_patches, c_feat), name='patch_flatten')(x)
 
    x = AddPositionEmbedding(n_patches=n_patches, embed_dim=c_feat,
                              name='patch_pos_add')(x)
 
    # Transformer encoder: global face context
    for i in range(4):
        x = hybrid_transformer_block(
            x,
            num_heads = 8,
            ff_dim    = 512,
            dropout   = DROP_ATT,
            name      = f'transformer_{i}'
        )
 
    x = layers.LayerNormalization(epsilon=1e-6, name='final_ln')(x)
    x = layers.GlobalAveragePooling1D(name='patch_avg')(x)
 
    x   = layers.BatchNormalization(name='head_bn')(x)
    x   = layers.Dropout(DROP_HEAD, name='drop_1')(x)
    x   = layers.Dense(256, activation='gelu',
                        kernel_regularizer=reg, name='dense_1')(x)
    x   = layers.BatchNormalization(name='dense_1_bn')(x)
    x   = layers.Dropout(DROP_HEAD * 0.6, name='drop_2')(x)
    out = layers.Dense(N_CLASSES, activation='softmax',
                        dtype='float32', name='predictions')(x)
 
    model = tf.keras.Model(inp, out, name=NAME)
    model.summary()
    print(f'\n  {NAME}: {model.count_params():,} total parameters')
    print(f'  Patch grid: {h_feat}×{w_feat} = {n_patches} tokens  |  dim: {c_feat}')
 
    def _cbs(tag, lr_max, warmup, total, patience, min_lr=1e-7):
        return [
            TerminateOnNaN(),
            tf.keras.callbacks.LearningRateScheduler(
                make_lr_schedule(lr_max, warmup, total, min_lr), verbose=1),
            EarlyStopping(monitor='val_accuracy', mode='max',
                          patience=patience, restore_best_weights=True,
                          verbose=1),
            ModelCheckpoint(
                os.path.join(SAVE_DIR, f'{tag}_best.keras'),
                monitor='val_accuracy', mode='max',
                save_best_only=True, verbose=1),
            CSVLogger(os.path.join(SAVE_DIR, f'{tag}_log.csv')),
        ]
 
    # ── Phase 1 ───────────────────────────────────────────────
    print(f'\n{"="*60}')
    print(f'  {NAME} | Phase 1 — AdamW  peak LR={P1_PEAK_LR}  {P1_EPOCHS} ep')
    print(f'{"="*60}')
    model.compile(
        optimizer = tf.keras.optimizers.AdamW(
                        learning_rate = P1_PEAK_LR,
                        weight_decay  = 2e-4,
                        clipnorm      = 1.0
                    ),
        loss    = smooth_loss,
        metrics = ['accuracy']
    )
    h1 = model.fit(
        mixed_generator_ds(train_ds, cw_vals, alpha=0.3, cutmix_prob=0.5),
        steps_per_epoch = STEPS_PER_EPOCH,
        validation_data = (X_val, y_val),
        epochs          = P1_EPOCHS,
        callbacks       = _cbs(NAME + '_p1', P1_PEAK_LR, P1_WARMUP,
                                P1_EPOCHS, P1_PATIENCE_ES, min_lr=1e-6)
    )
 
    # ── Phase 2 ───────────────────────────────────────────────
    print(f'\n{"="*60}')
    print(f'  {NAME} | Phase 2 — AdamW  peak LR={P2_PEAK_LR}  {P2_EPOCHS} ep')
    print(f'{"="*60}')
    model.compile(
        optimizer = tf.keras.optimizers.AdamW(
                        learning_rate = P2_PEAK_LR,
                        weight_decay  = 1e-4,
                        clipnorm      = 1.0
                    ),
        loss    = smooth_loss,
        metrics = ['accuracy']
    )
    h2 = model.fit(
        mixed_generator_ds(train_ds, cw_vals, alpha=0.3, cutmix_prob=0.5),
        steps_per_epoch = STEPS_PER_EPOCH,
        validation_data = (X_val, y_val),
        epochs          = P2_EPOCHS,
        callbacks       = _cbs(NAME + '_p2', P2_PEAK_LR, P2_WARMUP,
                                P2_EPOCHS, P2_PATIENCE_ES, min_lr=1e-7)
    )
 
    plot_history(h1, h2, NAME)
 
    best_p1 = max(h1.history['val_accuracy'])
    best_p2 = max(h2.history['val_accuracy'])
    best    = max(best_p1, best_p2)
    print(f'\n  Phase 1 best: {best_p1*100:.2f}%  |  Phase 2 best: {best_p2*100:.2f}%')
    print(f'  Overall best: {best*100:.2f}%')
 
    print('\n  Running TTA (6 passes)...')
    tta_preds = predict_with_tta(model, X_val, n_aug=6)
    tta_acc   = np.mean(np.argmax(tta_preds, axis=1) == np.argmax(y_val, axis=1))
    print(f'  {NAME} TTA val accuracy: {tta_acc*100:.2f}%')
 
    model.save(os.path.join(SAVE_DIR, f'{NAME}_final.keras'))
    return model, best, tta_acc
 
 
model_hybrid, acc_hybrid, acc_hybrid_tta = train_model_hybrid()

# evaluation

In [ ]:
import gc
import re
import os
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications.efficientnet import preprocess_input as ef_preprocess
from tensorflow.keras.applications.resnet_v2 import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.xception import preprocess_input as xception_preprocess

# ── Load ALL models from directory ────────────────────────────
MODELS_DIR = '//kaggle/input/models/ahmedjimmy101/emotion-models/other/default/5'

all_files = sorted([
    f for f in os.listdir(MODELS_DIR)
    if f.endswith('.keras')
    and '_p1_' not in f
    and '_p2_' not in f
])

print(f'Found {len(all_files)} model files:')
for f in all_files:
    print(f'  {f}')

# Map each model filename to its correct preprocess function
PREPROCESS_MAP = {
    'EfficientNetB3'     : ef_preprocess,
    'EfficientNetB2_CBAM': ef_preprocess,
    'ResNet50V2'         : resnet_preprocess,
    'Xception'           : xception_preprocess,
    'CustomDeepCNN'      : None,
    'CustomResNet'       : None,
    'CustomSEResNet'     : None,
}

all_models = []
all_names  = []

for filename in all_files:
    path = os.path.join(MODELS_DIR, filename)
    print(f'\nLoading: {filename}')

    # Find which preprocess_input this model needs
    preprocess_fn = None
    for key, fn in PREPROCESS_MAP.items():
        if key in filename:
            preprocess_fn = fn
            break

    # Pass preprocess function directly via custom_objects
    if preprocess_fn is not None:
        custom_objects = {'preprocess_input': preprocess_fn}
    else:
        custom_objects = {}

    model = keras.models.load_model(path, custom_objects=custom_objects, safe_mode=False)
    all_models.append(model)
    all_names.append(re.sub(r'\.keras$', '', filename))
    print(f'  OK — preprocess: {preprocess_fn.__module__ if preprocess_fn else "built-in Rescaling"}')

print(f'\nTotal models loaded: {len(all_models)}')

# ── Weights ───────────────────────────────────────────────────
val_scores = {
    'CustomDeepCNN_best'        : 0.8014,
    'CustomResNet_best'         : 0.8133,
    'CustomSEResNet_best'       : 0.8163,
    'EfficientNetB3_final'      : 0.8414,
    'ResNet50V2_final'          : 0.8502,
    'Xception_final'            : 0.8406,
    # 'EfficientNetB2_CBAM_final' : 0.8129,
}
all_scores = [val_scores.get(name, 1.0) for name in all_names]

weights = np.array(all_scores) / sum(all_scores)
print('\nEnsemble weights:')
for name, w in zip(all_names, weights):
    print(f'  {name:<35}  weight={w:.4f}')

# ── TTA: 5 views for robustness ───────────────────────────────
def tta_predict(model, img):
    expected_shape = model.input_shape
    expected_size  = expected_shape[1]
    expected_ch    = expected_shape[3]

    # ── Resize if needed ──────────────────────────────────────
    if expected_size != img.shape[1]:
        img = np.array([
            cv2.resize(img[0], (expected_size, expected_size),
                       interpolation=cv2.INTER_AREA)
        ], dtype=np.float32)

    # ── Convert channels if needed ────────────────────────────
    if expected_ch == 1 and img.shape[-1] == 3:
        img = np.mean(img, axis=-1, keepdims=True)
    elif expected_ch == 3 and img.shape[-1] == 1:
        img = np.repeat(img, 3, axis=-1)

    # ── Build TTA views ───────────────────────────────────────
    orig   = img                                        # (1, H, W, C)
    hflip  = img[:, :, ::-1, :]
    br_up  = np.clip(img * 1.10, 0, 255)
    br_dn  = np.clip(img * 0.90, 0, 255)

    crop   = int(expected_size * 0.90)
    pad    = (expected_size - crop) // 2
    z      = img[0, pad:pad+crop, pad:pad+crop, :]      # (crop, crop, C)
    z      = cv2.resize(z, (expected_size, expected_size),
                        interpolation=cv2.INTER_AREA)   # (H, W, C)

    # ── Fix: ensure zoomed is always 4D ──────────────────────
    if z.ndim == 2:
        z = z[:, :, np.newaxis]   # grayscale lost channel dim
    zoomed = z[np.newaxis]        # (1, H, W, C)

    views = np.concatenate([orig, hflip, br_up, br_dn, zoomed], axis=0)
    preds = model.predict(views, verbose=0, batch_size=len(views))
    return np.mean(preds, axis=0, keepdims=True)
# ── Load and sort test files ──────────────────────────────────
def sort_key(fname):
    match = re.search(r'\d+', fname)
    return int(match.group()) if match else fname

test_files = sorted(os.listdir(TEST_DIR), key=sort_key)
print(f'\nTest files found: {len(test_files)}')

# ── Predict ───────────────────────────────────────────────────
ids, labels, confidences = [], [], []

for i, fname in enumerate(test_files):

    path = os.path.join(TEST_DIR, fname)
    img  = cv2.imread(path)

    if img is None:
        print(f'  WARNING: could not read {fname} — skipping')
        continue

    if len(img.shape) == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    elif img.shape[2] == 1:
        img = cv2.cvtColor(img[:,:,0], cv2.COLOR_GRAY2RGB)
    else:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    img = np.array(img, dtype=np.float32)[np.newaxis]

    # ── Weighted ensemble + TTA ───────────────────────────────
    ensemble_pred = np.zeros((1, N_CLASSES), dtype=np.float32)
    for model, w in zip(all_models, weights):
        ensemble_pred += w * tta_predict(model, img)

    pred_class = int(np.argmax(ensemble_pred))
    confidence = float(ensemble_pred[0, pred_class])

    ids.append(re.sub(r'\.[^.]+$', '', fname))
    labels.append(CLASSES[pred_class])
    confidences.append(round(confidence, 4))

    if (i + 1) % 100 == 0:
        print(f'  Processed {i+1}/{len(test_files)}...')

# ── Build submission ──────────────────────────────────────────
submission = pd.DataFrame({
    'ID'    : ids,
    'Label' : labels
})

submission.to_csv('/kaggle/working/submission.csv', index=False)

# ── Report ────────────────────────────────────────────────────
print(f'\n{"="*50}')
print(f'  Submission saved: {len(submission)} predictions')
print(f'{"="*50}')

print('\nLabel distribution:')
dist = submission['Label'].value_counts()
for label, count in dist.items():
    pct = count / len(submission) * 100
    bar = '█' * int(pct / 2)
    print(f'  {label:<12} {count:>5}  {pct:>5.1f}%  {bar}')

print('\nFirst 10 predictions:')
print(submission.head(10).to_string(index=False))

# ── Confidence stats ──────────────────────────────────────────
conf_arr = np.array(confidences)
print(f'\nConfidence stats:')
print(f'  Mean   : {conf_arr.mean():.3f}')
print(f'  Min    : {conf_arr.min():.3f}')
print(f'  Max    : {conf_arr.max():.3f}')
print(f'  < 0.50 : {(conf_arr < 0.50).sum()} uncertain predictions')
print(f'  < 0.40 : {(conf_arr < 0.40).sum()} very uncertain predictions')

# ── Free memory ───────────────────────────────────────────────
gc.collect()

In [ ]:
import pandas as pd
df= pd.read_csv('/kaggle/working/submission.csv')
df.head()